# B. rapa Input Data Preprocessing, Updated for Larger Dataset

In [1]:
# Imports

import pandas as pd
import os
from pathlib import Path

In [2]:
# =============================================================================
# Utility Functions
# =============================================================================

# NaN-exempt columns: NaN values in these columns are biologically meaningful
# and are RETAINED rather than triggering pair dropout.
#
# tau_LF, tau_MF1, tau_MF2:
#     Tau is mathematically undefined (not missing) for genes with FPKM = 0
#     across all tissues. Retaining NaN preserves this biological distinction.
#
# ACR columns are added to this set in the ACR feature section below, after
# those column names are known. ACR NaNs arise from genes within 500 bp of a
# chromosome boundary where the ACR window cannot be fully extracted — a
# positional artifact, not a data quality issue.

NAN_EXEMPT_COLS = {
    'tau_LF',
    'tau_MF1',
    'tau_MF2',
    'upstream_distance_LF',
    'upstream_distance_MF1',
    'upstream_distance_MF2',
    'downstream_distance_LF',
    'downstream_distance_MF1',
    'downstream_distance_MF2',
}


def dropout_preview(lf_mf1: pd.DataFrame,
                    lf_mf2: pd.DataFrame,
                    mf1_mf2: pd.DataFrame,
                    nan_exempt_cols: set = None,
                    label: str = "") -> None:
    """
    Report how many rows would be dropped from each pairwise dataframe due to
    NaN values in required feature columns.

    Call this after merging each feature category to track cumulative data
    retention as features are added. The final call should be made after all
    features are merged to confirm the number of pairs available for modelling.

    Parameters
    ----------
    lf_mf1 : pd.DataFrame
    lf_mf2 : pd.DataFrame
    mf1_mf2 : pd.DataFrame
    nan_exempt_cols : set, optional
        Column names whose NaN values are biologically meaningful and should
        not trigger dropout. Defaults to the module-level NAN_EXEMPT_COLS set.
        Pass an explicit set to override (e.g. during testing).
    label : str, optional
        Short description of which feature was most recently added, printed in
        the header for readability when this function is called multiple times.
        Example: "after GC content merge"
    """
    if nan_exempt_cols is None:
        nan_exempt_cols = NAN_EXEMPT_COLS

    # Columns that are never dropout candidates regardless of NaN status.
    # These are structural identifiers and metadata, not features.
    metadata_cols = {
        'gene_num', 'ara_paralog', 'AKBr', 'ACK_block',
        'LF', 'MF1', 'MF2'
    }

    header = f"=== Pair-wise Dropout Preview"
    if label:
        header += f" ({label})"
    header += " ==="
    print(header)
    print(f"NaN-exempt columns: {sorted(nan_exempt_cols)}\n")

    for df_pair, pair_label in [
        (lf_mf1,  'lf_mf1_pairs'),
        (lf_mf2,  'lf_mf2_pairs'),
        (mf1_mf2, 'mf1_mf2_pairs'),
    ]:
        # Feature columns only: exclude metadata and exempt columns
        dropout_cols = [
            col for col in df_pair.columns
            if col not in metadata_cols
            and col not in nan_exempt_cols
        ]

        would_drop = df_pair[dropout_cols].isna().any(axis=1)
        n_total = len(df_pair)
        n_drop  = would_drop.sum()
        n_keep  = n_total - n_drop

        print(f"  {pair_label}:")
        print(f"    Total pairs   : {n_total}")
        print(f"    Pairs to drop : {n_drop} ({100 * n_drop / n_total:.1f}%)")
        print(f"    Pairs to keep : {n_keep} ({100 * n_keep / n_total:.1f}%)")

        # Only print the column breakdown when there are drops to explain.
        # This keeps the output clean when all data is present.
        if n_drop > 0:
            col_nan_counts = (
                df_pair.loc[would_drop, dropout_cols]
                .isna()
                .sum()
            )
            col_nan_counts = col_nan_counts[col_nan_counts > 0].sort_values(
                ascending=False
            )
            print(f"    Columns driving dropout:")
            for col, count in col_nan_counts.items():
                print(f"      {col}: {count} affected pairs")
        print()

    print("Note: counts will change as more features are added. "
          "Run after each feature category merge.")

In [3]:
# Load raw master file

# Load master B. rapa syntenic gene table
# Source: Brapa_3genomes_tPCK.confident
# This is a complete export from the BRAD database containing all ancestral karyotype
# positions. It is a superset of the Cheng et al. (2012) supplemental table, which
# includes only positions where at least one Arabidopsis ortholog is annotated.
# The two extra row categories present here but not in the paper supplement are:
#   (1) Positions where ara_paralog == "-" (no Arabidopsis ortholog detectable)
#   (2) Rows where the Arabidopsis ID carries a "-TA" suffix (tandem array flags)
# Neither category contains Bra gene IDs and both are filtered below.
# Column definitions:
#   gene_num    : sequential row index from source file (metadata only)
#   ara_paralog : Arabidopsis thaliana syntenic ortholog ID, or "-" if absent
#   AKBr        : ancestral karyotype chromosome (AKBr1–AKBr7)
#   ACK_block   : ancestral karyotype block letter (A–X, per Schranz et al. 2006)
#   LF          : B. rapa gene ID in the least-fractionated subgenome, or "-" if lost
#   MF1         : B. rapa gene ID in more-fractionated subgenome 1, or "-" if lost
#   MF2         : B. rapa gene ID in more-fractionated subgenome 2, or "-" if lost

# Data directory: download raw input data from Zenodo (see inputData/README.md)
# and extract into this location before running this notebook.
DATA_DIR = Path("../inputData/brapa_inputData")

brapa_master = pd.read_csv(
    DATA_DIR / "Brapa_3genomes_tPCK.confident",
    sep='\t',
    header=None,
    names=['gene_num', 'ara_paralog', 'AKBr', 'ACK_block', 'LF', 'MF1', 'MF2']
)

print(f"Raw file shape: {brapa_master.shape}")
display(brapa_master.head())

Raw file shape: (32749, 7)


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2
0,1,-,AKBr1,A,-,-,-
1,2,AT1G01010,AKBr1,A,Bra033296,-,-
2,3,AT1G01020,AKBr1,A,Bra033295,-,-
3,4,AT1G01030,AKBr1,A,Bra033294,-,-
4,5,AT1G01040,AKBr1,A,Bra033293,-,-


In [4]:
# Sentinel validation pre-check

print("=== Master File Sentinel Validation ===\n")

# Only "-" and real gene IDs are permitted in the Bra gene columns.
# NaNs, empty strings, or any other sentinels indicate a file format problem
# that must be resolved before proceeding.

warnings = 0

for col in ['LF', 'MF1', 'MF2']:
    # Check for NaN
    n_nan = brapa_master[col].isna().sum()
    if n_nan > 0:
        raise ValueError(
            f"CRITICAL: {n_nan} NaN values detected in column '{col}'. "
            f"Only gene IDs or '-' are permitted. Inspect source file before proceeding."
        )

    # Check for empty strings
    n_empty = (brapa_master[col] == '').sum()
    if n_empty > 0:
        raise ValueError(
            f"CRITICAL: {n_empty} empty string values detected in column '{col}'. "
            f"Only gene IDs or '-' are permitted. Inspect source file before proceeding."
        )

    # Confirm all non-dash values look like Bra gene IDs
    non_dash = brapa_master[col][brapa_master[col] != '-']
    unexpected = non_dash[~non_dash.str.startswith('Bra')]
    if len(unexpected) > 0:
        raise ValueError(
            f"CRITICAL: Unexpected values in column '{col}' that are neither '-' "
            f"nor a 'Bra' gene ID:\n{unexpected.unique().tolist()}\n"
            f"Inspect source file before proceeding."
        )

    print(f"  ✓ {col}: all values are '-' or valid Bra gene IDs")

# Check ara_paralog: permit "-", "AT*", and "AT*-TA" only
ara_non_dash = brapa_master['ara_paralog'][brapa_master['ara_paralog'] != '-']
unexpected_ara = ara_non_dash[
    ~ara_non_dash.str.startswith('AT') &
    ~ara_non_dash.str.startswith('at')
]
if len(unexpected_ara) > 0:
    raise ValueError(
        f"CRITICAL: Unexpected values in 'ara_paralog':\n{unexpected_ara.unique().tolist()}"
    )
print(f"  ✓ ara_paralog: all values are '-' or valid Arabidopsis gene IDs\n")
print(f"=== Sentinel validation passed ===")

=== Master File Sentinel Validation ===

  ✓ LF: all values are '-' or valid Bra gene IDs
  ✓ MF1: all values are '-' or valid Bra gene IDs
  ✓ MF2: all values are '-' or valid Bra gene IDs
  ✓ ara_paralog: all values are '-' or valid Arabidopsis gene IDs

=== Sentinel validation passed ===


In [5]:
# Filter extra rows, strip -TA suffixes, handle duplicates

print("=== Master File Cleaning ===\n")
n_raw = len(brapa_master)

# --- Step 1: Remove rows where ara_paralog == "-" ---
# These are ancestral karyotype positions with no detectable Arabidopsis ortholog.
# None carry Bra gene IDs and none are used in any downstream merge.
no_ortholog_mask = brapa_master['ara_paralog'] == '-'
n_no_ortholog = no_ortholog_mask.sum()
brapa_master = brapa_master[~no_ortholog_mask].reset_index(drop=True)
print(f"Removed {n_no_ortholog} rows where ara_paralog == '-' (no Arabidopsis ortholog).")

# --- Step 2: Remove rows where ara_paralog carries a -TA suffix ---
# These are tandem gene array flags from the BRAD database export.
# Per Cheng et al. (2012) Methods, tandem arrays were excluded from the high-confidence
# homoeolog set because they change copy number rapidly and complicate expression analysis.
# Confirmed: none of these rows carry Bra gene IDs in LF, MF1, or MF2.
ta_mask = brapa_master['ara_paralog'].str.endswith('-TA', na=False)
n_ta = ta_mask.sum()
brapa_master = brapa_master[~ta_mask].reset_index(drop=True)
print(f"Removed {n_ta} rows with '-TA' tandem array flags on ara_paralog.")

# --- Step 3: Strip -TA suffix from Bra gene columns as a defensive check ---
# Confirmed absent from LF, MF1, MF2 in this file, but applied defensively
# in case upstream processing ever reintroduces them.
for col in ['LF', 'MF1', 'MF2']:
    ta_bra = brapa_master[col].str.endswith('-TA', na=False)
    if ta_bra.any():
        print(f"  WARNING: {ta_bra.sum()} '-TA' suffixes found in {col} — stripping.")
        brapa_master.loc[ta_bra, col] = brapa_master.loc[ta_bra, col].str.split('-').str[0]
    else:
        print(f"  ✓ {col}: no '-TA' suffixes present")

# --- Step 4: Handle duplicate rows ---
# One exact duplicate is known to exist in this file (AT5G67640, last two rows).
# Any duplicate detected here is treated as a source file artifact, not a processing error.
dup_mask = brapa_master.duplicated()
n_dups = dup_mask.sum()
if n_dups > 0:
    dup_ids = brapa_master.loc[dup_mask, 'ara_paralog'].tolist()
    print(
        f"\n  WARNING: {n_dups} duplicate row(s) detected and will be dropped.\n"
        f"  Affected ara_paralog IDs: {dup_ids}\n"
        f"  These are source file artifacts. If this count exceeds 1, inspect the file."
    )
    brapa_master = brapa_master[~dup_mask].reset_index(drop=True)
else:
    print("\n  ✓ No duplicate rows detected")

print(f"\nRows before cleaning: {n_raw}")
print(f"Rows after cleaning:  {len(brapa_master)}")

=== Master File Cleaning ===

Removed 8019 rows where ara_paralog == '-' (no Arabidopsis ortholog).
Removed 1569 rows with '-TA' tandem array flags on ara_paralog.
  ✓ LF: no '-TA' suffixes present
  ✓ MF1: no '-TA' suffixes present
  ✓ MF2: no '-TA' suffixes present

  ✓ No duplicate rows detected

Rows before cleaning: 32749
Rows after cleaning:  23161


In [6]:
# Verify doublet and triplet counts against Cheng et al. (2016)

print("=== Doublet and Triplet Count Verification ===\n")
print("Expected counts from Cheng et al. (2016) and confirmed against source data:")
print("  LF–MF1 doublets : 4,294")
print("  LF–MF2 doublets : 3,666")
print("  MF1–MF2 doublets: 2,504")
print("  Triplets        : 1,675\n")

lf_mf1_count  = ((brapa_master['LF'] != '-') & (brapa_master['MF1'] != '-')).sum()
lf_mf2_count  = ((brapa_master['LF'] != '-') & (brapa_master['MF2'] != '-')).sum()
mf1_mf2_count = ((brapa_master['MF1'] != '-') & (brapa_master['MF2'] != '-')).sum()
triplet_count = (
    (brapa_master['LF'] != '-') &
    (brapa_master['MF1'] != '-') &
    (brapa_master['MF2'] != '-')
).sum()

assert lf_mf1_count  == 4294, f"CRITICAL: LF–MF1 doublet count is {lf_mf1_count}, expected 4,294."
assert lf_mf2_count  == 3666, f"CRITICAL: LF–MF2 doublet count is {lf_mf2_count}, expected 3,666."
assert mf1_mf2_count == 2504, f"CRITICAL: MF1–MF2 doublet count is {mf1_mf2_count}, expected 2,504."
assert triplet_count == 1675, f"CRITICAL: Triplet count is {triplet_count}, expected 1,675."

print(f"  ✓ LF–MF1 doublets : {lf_mf1_count}")
print(f"  ✓ LF–MF2 doublets : {lf_mf2_count}")
print(f"  ✓ MF1–MF2 doublets: {mf1_mf2_count}")
print(f"  ✓ Triplets        : {triplet_count}")
print(f"\n=== All counts verified ===")

=== Doublet and Triplet Count Verification ===

Expected counts from Cheng et al. (2016) and confirmed against source data:
  LF–MF1 doublets : 4,294
  LF–MF2 doublets : 3,666
  MF1–MF2 doublets: 2,504
  Triplets        : 1,675

  ✓ LF–MF1 doublets : 4294
  ✓ LF–MF2 doublets : 3666
  ✓ MF1–MF2 doublets: 2504
  ✓ Triplets        : 1675

=== All counts verified ===


In [7]:
# Display final master dataframe

print(f"Final brapa_master shape: {brapa_master.shape}")
display(brapa_master)

Final brapa_master shape: (23161, 7)


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2
0,2,AT1G01010,AKBr1,A,Bra033296,-,-
1,3,AT1G01020,AKBr1,A,Bra033295,-,-
2,4,AT1G01030,AKBr1,A,Bra033294,-,-
3,5,AT1G01040,AKBr1,A,Bra033293,-,-
4,6,AT1G01050,AKBr1,A,Bra033292,-,Bra032616
...,...,...,...,...,...,...,...
23156,32745,AT5G67610,AKBr7,X,Bra024435,-,-
23157,32746,AT5G67620,AKBr7,X,Bra024434,Bra031842,Bra037835
23158,32747,AT5G67630,AKBr7,X,Bra024433,-,Bra037834
23159,32748,AT5G67640,AKBr7,X,Bra024432,-,-


In [8]:
# Create pairwise pair dataframes

# === Pairwise Gene Pair Dataframes ===
#
# brapa_master is filtered into three pairwise dataframes, one per subgenome
# comparison. Each row in these dataframes represents a single gene doublet —
# two homoeologous genes retained in two different subgenomes.
#
# These three dataframes are the direct brassica equivalent of maize_df in the
# maize preprocessing notebook. All feature data will be merged into these
# dataframes sequentially in the sections that follow.
#
# Note on gene membership across dataframes:
#   Genes retained in all three subgenomes (triplets, n=1675) will appear in
#   all three pairwise dataframes — once as an LF-MF1 pair, once as an LF-MF2
#   pair, and once as an MF1-MF2 pair. This is correct by design. Each
#   pairwise comparison is treated as an independent model.
#
# Metadata columns (gene_num, ara_paralog, AKBr, ACK_block) are retained in
# each dataframe for traceability but will not be used as model features.

lf_mf1_pairs = brapa_master[
    (brapa_master['LF'] != '-') & (brapa_master['MF1'] != '-')
].copy().reset_index(drop=True)

lf_mf2_pairs = brapa_master[
    (brapa_master['LF'] != '-') & (brapa_master['MF2'] != '-')
].copy().reset_index(drop=True)

mf1_mf2_pairs = brapa_master[
    (brapa_master['MF1'] != '-') & (brapa_master['MF2'] != '-')
].copy().reset_index(drop=True)

print(f"lf_mf1_pairs  shape: {lf_mf1_pairs.shape}")
print(f"lf_mf2_pairs  shape: {lf_mf2_pairs.shape}")
print(f"mf1_mf2_pairs shape: {mf1_mf2_pairs.shape}")

lf_mf1_pairs  shape: (4294, 7)
lf_mf2_pairs  shape: (3666, 7)
mf1_mf2_pairs shape: (2504, 7)


In [9]:
# Assert row counts

print("=== Pairwise Pair Count Assertions ===\n")

assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows, expected 2504."

print(f"  ✓ lf_mf1_pairs  : {len(lf_mf1_pairs)} rows")
print(f"  ✓ lf_mf2_pairs  : {len(lf_mf2_pairs)} rows")
print(f"  ✓ mf1_mf2_pairs : {len(mf1_mf2_pairs)} rows")
print(f"\n=== All pair counts verified ===")

=== Pairwise Pair Count Assertions ===

  ✓ lf_mf1_pairs  : 4294 rows
  ✓ lf_mf2_pairs  : 3666 rows
  ✓ mf1_mf2_pairs : 2504 rows

=== All pair counts verified ===


In [10]:
# Within-pair gene ID overlap check

print("=== Within-Pair Gene ID Overlap Checks ===\n")
print("Verifying that no gene ID appears in both subgenome columns of the same pair.")
print("A gene cannot be its own pair partner.\n")

# lf_mf1_pairs: LF and MF1 columns must be disjoint
lf_mf1_overlap = set(lf_mf1_pairs['LF']) & set(lf_mf1_pairs['MF1'])
if lf_mf1_overlap:
    raise ValueError(
        f"CRITICAL: {len(lf_mf1_overlap)} gene IDs appear in both LF and MF1 "
        f"columns of lf_mf1_pairs. WGD label assignment will be incorrect.\n"
        f"Affected IDs: {sorted(list(lf_mf1_overlap))}"
    )
print(f"  ✓ lf_mf1_pairs  : no overlap between LF and MF1 gene IDs")

# lf_mf2_pairs: LF and MF2 columns must be disjoint
lf_mf2_overlap = set(lf_mf2_pairs['LF']) & set(lf_mf2_pairs['MF2'])
if lf_mf2_overlap:
    raise ValueError(
        f"CRITICAL: {len(lf_mf2_overlap)} gene IDs appear in both LF and MF2 "
        f"columns of lf_mf2_pairs. WGD label assignment will be incorrect.\n"
        f"Affected IDs: {sorted(list(lf_mf2_overlap))}"
    )
print(f"  ✓ lf_mf2_pairs  : no overlap between LF and MF2 gene IDs")

# mf1_mf2_pairs: MF1 and MF2 columns must be disjoint
mf1_mf2_overlap = set(mf1_mf2_pairs['MF1']) & set(mf1_mf2_pairs['MF2'])
if mf1_mf2_overlap:
    raise ValueError(
        f"CRITICAL: {len(mf1_mf2_overlap)} gene IDs appear in both MF1 and MF2 "
        f"columns of mf1_mf2_pairs. WGD label assignment will be incorrect.\n"
        f"Affected IDs: {sorted(list(mf1_mf2_overlap))}"
    )
print(f"  ✓ mf1_mf2_pairs : no overlap between MF1 and MF2 gene IDs")

print(f"\n=== All overlap checks passed ===")

=== Within-Pair Gene ID Overlap Checks ===

Verifying that no gene ID appears in both subgenome columns of the same pair.
A gene cannot be its own pair partner.

  ✓ lf_mf1_pairs  : no overlap between LF and MF1 gene IDs
  ✓ lf_mf2_pairs  : no overlap between LF and MF2 gene IDs
  ✓ mf1_mf2_pairs : no overlap between MF1 and MF2 gene IDs

=== All overlap checks passed ===


In [11]:
# Build master gene set and display

# Build the master set of all unique Bra gene IDs across all three pairwise
# dataframes. This is used in the pre-check cells of each feature section to
# verify coverage before any merge is attempted.

master_genes = (
    set(lf_mf1_pairs['LF'])  | set(lf_mf1_pairs['MF1'])  |
    set(lf_mf2_pairs['LF'])  | set(lf_mf2_pairs['MF2'])  |
    set(mf1_mf2_pairs['MF1'])| set(mf1_mf2_pairs['MF2'])
)

print(f"Total unique Bra gene IDs across all pairwise dataframes: {len(master_genes)}")
print()
display(lf_mf1_pairs.head())
display(lf_mf2_pairs.head())
display(mf1_mf2_pairs.head())

Total unique Bra gene IDs across all pairwise dataframes: 15903



,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2
0,62,AT1G01520,AKBr1,A,Bra033257,Bra030496,-
1,167,AT1G02400,AKBr1,A,Bra033324,Bra030500,-
2,171,AT1G02410,AKBr1,A,Bra033325,Bra030501,Bra032603
3,186,AT1G02560,AKBr1,A,Bra033332,Bra030504,-
4,193,AT1G02610,AKBr1,A,Bra033335,Bra030505,-


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2
0,6,AT1G01050,AKBr1,A,Bra033292,-,Bra032616
1,14,AT1G01090,AKBr1,A,Bra033286,-,Bra032619
2,15,AT1G01100,AKBr1,A,Bra033285,-,Bra032620
3,18,AT1G01120,AKBr1,A,Bra033283,-,Bra032621
4,22,AT1G01160,AKBr1,A,Bra033281,-,Bra032623


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2
0,171,AT1G02410,AKBr1,A,Bra033325,Bra030501,Bra032603
1,198,AT1G02660,AKBr1,A,-,Bra030507,Bra032591
2,213,AT1G02780,AKBr1,A,Bra033346,Bra030509,Bra032586
3,214,AT1G02790,AKBr1,A,Bra033347,Bra030510,Bra032585
4,233,AT1G02970,AKBr1,A,-,Bra030517,Bra032577


## Histone Markers

In [12]:
histone_file_paths = [
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_H3K27ac_counts_log_ave', 'H3K27ac_down'),
    ('Brapa_gene_v1.5_genes_2kbup_40wins_H3K27ac_counts_log_ave',   'H3K27ac_up'),
    ('Brapa_gene_v1.5_genes_body_40wins_H3K27ac_counts_log_ave',    'H3K27ac_genebody'),

    ('Brapa_gene_v1.5_genes_2kbdown_40wins_H3K27me3_counts_log_ave', 'H3K27me3_down'),
    ('Brapa_gene_v1.5_genes_2kbup_40wins_H3K27me3_counts_log_ave',   'H3K27me3_up'),
    ('Brapa_gene_v1.5_genes_body_40wins_H3K27me3_counts_log_ave',    'H3K27me3_genebody'),

    ('Brapa_gene_v1.5_genes_2kbdown_40wins_H3K4me3_counts_log_ave',  'H3K4me3_down'),
    ('Brapa_gene_v1.5_genes_2kbup_40wins_H3K4me3_counts_log_ave',    'H3K4me3_up'),
    ('Brapa_gene_v1.5_genes_body_40wins_H3K4me3_counts_log_ave',     'H3K4me3_genebody'),
]

### Histone Data Pre-check

In [13]:
print("=== Histone Data Pre-check ===\n")
print(f"Total unique genes in master doublet set: {len(master_genes)}")
print(f"Expected files: {len(histone_file_paths)} "
      f"(3 markers × 3 regions: down, up, genebody)\n")

total_warnings = 0

for file_path, col_name in histone_file_paths:
    df_tmp = pd.read_csv(
        DATA_DIR / file_path, delimiter='\t', header=None,
        names=['raw_id', col_name]
    )
    df_tmp['gene_id'] = df_tmp['raw_id'].str.split('_').str[0]

    # Duplicate check — hard stop if duplicates detected
    duplicates = df_tmp['gene_id'].duplicated()
    if duplicates.any():
        dup_ids = df_tmp.loc[duplicates, 'gene_id'].unique().tolist()
        raise ValueError(
            f"CRITICAL: {len(dup_ids)} duplicate gene IDs detected in {col_name} "
            f"after stripping suffix. This must be resolved before proceeding.\n"
            f"Affected IDs: {dup_ids[:10]}"
        )

    # Coverage check against master doublet gene set
    file_genes = set(df_tmp['gene_id'])
    absent_genes = master_genes - file_genes
    if absent_genes:
        print(f"  WARNING: {col_name}: {len(absent_genes)} master genes absent.")
        print(f"    Absent IDs: {sorted(list(absent_genes))[:10]}"
              f"{'...' if len(absent_genes) > 10 else ''}")
        total_warnings += 1
    else:
        print(f"  ✓  {col_name}: all master genes present")

print(f"\n=== Pre-check complete. Total warnings: {total_warnings} ===")
print("NaNs will be introduced for any absent genes during merge.")

=== Histone Data Pre-check ===

Total unique genes in master doublet set: 15903
Expected files: 9 (3 markers × 3 regions: down, up, genebody)

  ✓  H3K27ac_down: all master genes present
  ✓  H3K27ac_up: all master genes present
  ✓  H3K27ac_genebody: all master genes present
  ✓  H3K27me3_down: all master genes present
  ✓  H3K27me3_up: all master genes present
  ✓  H3K27me3_genebody: all master genes present
  ✓  H3K4me3_down: all master genes present
  ✓  H3K4me3_up: all master genes present
  ✓  H3K4me3_genebody: all master genes present

=== Pre-check complete. Total warnings: 0 ===
NaNs will be introduced for any absent genes during merge.


### Histone Processing and Merge

In [14]:
print("=== Histone Data Processing and Merge ===\n")

# --- Build single wide histone dataframe indexed by gene_id ---
# Identical approach to the maize notebook. Each file contributes one column.
# Deduplication is applied defensively even though no duplicates were
# detected in the pre-check.

histone_frames = []

for file_path, col_name in histone_file_paths:
    df_tmp = pd.read_csv(
        DATA_DIR / file_path, delimiter='\t', header=None,
        names=['raw_id', col_name]
    )
    df_tmp['gene_id'] = df_tmp['raw_id'].str.split('_').str[0]
    df_tmp = df_tmp.drop_duplicates(subset='gene_id', keep='first')
    df_tmp = df_tmp[['gene_id', col_name]].set_index('gene_id')
    histone_frames.append(df_tmp)

histone_wide = pd.concat(histone_frames, axis=1).reset_index()
histone_wide = histone_wide.rename(columns={'index': 'gene_id'})

print(f"Histone wide dataframe shape: {histone_wide.shape}")
print(f"Expected: {1 + len(histone_file_paths)} columns "
      f"(gene_id + {len(histone_file_paths)} histone features)\n")

# --- Merge into lf_mf1_pairs ---
histone_LF   = histone_wide.rename(columns={col: f'{col}_LF'  for _, col in histone_file_paths} | {'gene_id': 'LF'})
histone_MF1  = histone_wide.rename(columns={col: f'{col}_MF1' for _, col in histone_file_paths} | {'gene_id': 'MF1'})
histone_MF2  = histone_wide.rename(columns={col: f'{col}_MF2' for _, col in histone_file_paths} | {'gene_id': 'MF2'})

lf_mf1_pairs = pd.merge(lf_mf1_pairs, histone_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, histone_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, histone_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, histone_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, histone_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, histone_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after histone merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after histone merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after histone merge, expected 2504."

print("✓ Row counts verified after histone merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
histone_cols = [col for _, col in histone_file_paths]

print("\n--- NaN Report ---")
for df, label in [(lf_mf1_pairs, 'lf_mf1'), (lf_mf2_pairs, 'lf_mf2'), (mf1_mf2_pairs, 'mf1_mf2')]:
    suffixes = ['_LF', '_MF1'] if label == 'lf_mf1' else \
               ['_LF', '_MF2'] if label == 'lf_mf2' else \
               ['_MF1', '_MF2']
    cols_to_check = [f'{col}{suf}' for col in histone_cols for suf in suffixes]
    nan_report = df[cols_to_check].isna().sum()
    nan_report = nan_report[nan_report > 0]
    if len(nan_report) > 0:
        print(f"\n  {label} — NaNs detected (from genes absent in pre-check):")
        print(nan_report.to_string())
    else:
        print(f"  ✓  {label}: no NaNs in histone columns")

=== Histone Data Processing and Merge ===

Histone wide dataframe shape: (40915, 10)
Expected: 10 columns (gene_id + 9 histone features)

✓ Row counts verified after histone merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  ✓  lf_mf1: no NaNs in histone columns
  ✓  lf_mf2: no NaNs in histone columns
  ✓  mf1_mf2: no NaNs in histone columns


In [15]:
display(lf_mf1_pairs)
display(lf_mf2_pairs)
display(mf1_mf2_pairs)

,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2,H3K27ac_down_LF,H3K27ac_up_LF,H3K27ac_genebody_LF,...,H3K4me3_genebody_LF,H3K27ac_down_MF1,H3K27ac_up_MF1,H3K27ac_genebody_MF1,H3K27me3_down_MF1,H3K27me3_up_MF1,H3K27me3_genebody_MF1,H3K4me3_down_MF1,H3K4me3_up_MF1,H3K4me3_genebody_MF1
0,62,AT1G01520,AKBr1,A,Bra033257,Bra030496,-,-0.029402,0.129450,0.210577,...,0.980327,-0.147238,0.230721,-0.137395,0.134157,-0.198676,-0.318602,-0.122449,0.587409,0.137146
1,167,AT1G02400,AKBr1,A,Bra033324,Bra030500,-,-0.320951,-0.112204,0.020681,...,0.010274,-0.114324,-0.071605,-0.061795,0.011813,0.104503,0.441871,0.572138,-0.122492,0.183844
2,171,AT1G02410,AKBr1,A,Bra033325,Bra030501,Bra032603,-0.114827,-0.286421,-0.053887,...,-0.279706,0.470825,-0.070300,-0.253907,-0.265869,0.389337,-0.151289,0.713513,0.361324,0.216166
3,186,AT1G02560,AKBr1,A,Bra033332,Bra030504,-,-0.287275,0.138651,0.026742,...,0.598764,0.020459,0.245799,0.689229,-0.140267,-0.307222,-0.222425,-0.380946,0.422562,1.012648
4,193,AT1G02610,AKBr1,A,Bra033335,Bra030505,-,-0.048427,-0.208155,-0.066925,...,0.002056,0.110275,-0.051982,-0.390185,-0.286853,0.036925,0.150228,-0.146085,-0.055217,-0.311985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4289,32726,AT5G67450,AKBr7,X,Bra024448,Bra031834,Bra037845,-0.106294,-0.339133,1.247021,...,0.882149,-0.102093,0.107487,0.965260,0.021618,-0.298607,-0.307345,-0.040931,-0.241140,0.412201
4290,32730,AT5G67470,AKBr7,X,Bra024447,Bra031838,-,0.155034,0.220757,1.346085,...,1.250991,-0.041011,0.159204,0.179284,0.075616,0.068191,-0.126873,-0.044160,0.012657,0.316910
4291,32733,AT5G67500,AKBr7,X,Bra024444,Bra031839,Bra037844,0.082047,0.887733,0.332673,...,0.733248,-0.289157,-0.020635,0.016931,0.237025,0.019799,0.153675,-0.256121,-0.031348,-0.103755
4292,32744,AT5G67600,AKBr7,X,Bra024436,Bra031841,-,0.673964,0.091332,0.416186,...,-0.057788,-0.137196,-0.062940,0.357234,-0.190987,0.096357,0.109583,-0.400887,-0.097834,0.215275


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2,H3K27ac_down_LF,H3K27ac_up_LF,H3K27ac_genebody_LF,...,H3K4me3_genebody_LF,H3K27ac_down_MF2,H3K27ac_up_MF2,H3K27ac_genebody_MF2,H3K27me3_down_MF2,H3K27me3_up_MF2,H3K27me3_genebody_MF2,H3K4me3_down_MF2,H3K4me3_up_MF2,H3K4me3_genebody_MF2
0,6,AT1G01050,AKBr1,A,Bra033292,-,Bra032616,0.152907,0.053482,0.232187,...,0.274321,-0.123542,-0.151394,-0.033862,-0.077042,0.500652,-0.130707,-0.082304,0.004387,0.162470
1,14,AT1G01090,AKBr1,A,Bra033286,-,Bra032619,0.335198,0.403358,0.622701,...,0.729496,0.049420,0.501632,0.958124,0.262702,-0.011878,-0.193145,-0.146287,0.630318,1.299998
2,15,AT1G01100,AKBr1,A,Bra033285,-,Bra032620,0.527465,0.142465,0.548467,...,0.669238,0.761827,0.339531,0.688636,-0.118634,0.056749,-0.238517,1.172850,0.547377,0.623181
3,18,AT1G01120,AKBr1,A,Bra033283,-,Bra032621,-0.110294,-0.222441,0.362709,...,0.191544,0.437231,-0.047232,0.884527,-0.038384,0.045295,0.107978,0.501434,0.091247,1.149047
4,22,AT1G01160,AKBr1,A,Bra033281,-,Bra032623,0.642852,0.039367,0.506478,...,0.282665,-0.307353,-0.236397,0.619985,0.466342,0.515258,-0.147845,-0.195956,0.179157,0.545682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3661,32733,AT5G67500,AKBr7,X,Bra024444,Bra031839,Bra037844,0.082047,0.887733,0.332673,...,0.733248,-0.131785,0.117950,0.240217,0.345660,0.108583,-0.361457,-0.175804,0.167015,0.447581
3662,32735,AT5G67520,AKBr7,X,Bra024443,-,Bra037843,0.403138,0.138738,0.115879,...,0.479051,-0.138744,0.016738,-0.098686,0.495559,-0.153145,0.323767,-0.135369,-0.073618,-0.197717
3663,32741,AT5G67580,AKBr7,X,Bra024438,-,Bra037839,-0.207292,0.748044,-0.047762,...,-0.056988,0.473926,0.386417,0.135214,-0.207658,-0.044575,-0.156860,0.334466,0.486335,0.154650
3664,32746,AT5G67620,AKBr7,X,Bra024434,Bra031842,Bra037835,-0.193662,-0.003762,-0.310952,...,0.128592,0.162064,-0.082444,0.086121,0.094270,0.166783,0.129100,0.184122,-0.024709,-0.042998


,gene_num,ara_paralog,AKBr,ACK_block,LF,MF1,MF2,H3K27ac_down_MF1,H3K27ac_up_MF1,H3K27ac_genebody_MF1,...,H3K4me3_genebody_MF1,H3K27ac_down_MF2,H3K27ac_up_MF2,H3K27ac_genebody_MF2,H3K27me3_down_MF2,H3K27me3_up_MF2,H3K27me3_genebody_MF2,H3K4me3_down_MF2,H3K4me3_up_MF2,H3K4me3_genebody_MF2
0,171,AT1G02410,AKBr1,A,Bra033325,Bra030501,Bra032603,0.470825,-0.070300,-0.253907,...,0.216166,-0.039096,0.342579,0.205980,0.051390,0.243678,-0.078020,0.149604,0.326822,0.384786
1,198,AT1G02660,AKBr1,A,-,Bra030507,Bra032591,-0.208436,0.009664,0.451348,...,0.761264,0.098271,-0.044906,0.861170,0.003252,0.017143,-0.118207,-0.238005,0.002666,1.025007
2,213,AT1G02780,AKBr1,A,Bra033346,Bra030509,Bra032586,0.289012,0.144083,0.781203,...,1.285230,-0.278776,-0.113894,0.564209,-0.190930,0.172549,-0.355890,0.231156,0.017342,0.813892
3,214,AT1G02790,AKBr1,A,Bra033347,Bra030510,Bra032585,0.504354,-0.505092,0.005242,...,-0.142249,0.070714,-0.401885,-0.008905,0.128845,0.311430,0.277091,0.450459,-0.112560,-0.269395
4,233,AT1G02970,AKBr1,A,-,Bra030517,Bra032577,-0.052919,-0.214689,0.049127,...,0.264761,-0.144856,-0.146006,0.056535,0.603312,0.216657,-0.002402,-0.027999,-0.030802,0.186349
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499,32723,AT5G67420,AKBr7,X,Bra012164,Bra031833,Bra037847,0.107651,0.001602,1.921061,...,2.209888,0.164513,-0.010096,1.389347,-0.038720,0.020493,-0.043750,-0.219980,-0.021564,0.930780
2500,32726,AT5G67450,AKBr7,X,Bra024448,Bra031834,Bra037845,-0.102093,0.107487,0.965260,...,0.412201,0.100430,0.116000,1.256737,0.014018,0.102499,-0.120162,-0.125111,0.116127,1.297897
2501,32733,AT5G67500,AKBr7,X,Bra024444,Bra031839,Bra037844,-0.289157,-0.020635,0.016931,...,-0.103755,-0.131785,0.117950,0.240217,0.345660,0.108583,-0.361457,-0.175804,0.167015,0.447581
2502,32738,AT5G67550,AKBr7,X,-,Bra031840,Bra037842,-0.038918,-0.233520,-0.349070,...,-0.328687,-0.138427,0.173133,-0.355400,0.273087,0.071003,0.531924,-0.165738,-0.053738,-0.329534


In [16]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after histone merge")

=== Pair-wise Dropout Preview (after expression merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 0 (0.0%)
    Pairs to keep : 4294 (100.0%)

  lf_mf2_pairs:
    Total pairs   : 3666
    Pairs to drop : 0 (0.0%)
    Pairs to keep : 3666 (100.0%)

  mf1_mf2_pairs:
    Total pairs   : 2504
    Pairs to drop : 0 (0.0%)
    Pairs to keep : 2504 (100.0%)

Note: counts will change as more features are added. Run after each feature category merge.


## Expression Data 

In [17]:
# Load the preprocessed B. rapa expression feature matrix.
# This file is produced by brapa_expression_log2_tau_processing.py,
# which ingests raw FPKM values from GEO accession GSE43245
# (Bai et al. 2014, BMC Genomics 14:689) and computes two features:
#
#   Log2_Average : mean of log2(FPKM + 1) across 8 tissue columns
#                  (Callus, Flower, Leaf1, Leaf2, Root1, Root2, Silique, Stem)
#
#   Tau_Index    : tissue specificity index calculated from 6 biologically
#                  distinct conditions after averaging replicates:
#                  Leaf = mean(Leaf1, Leaf2), Root = mean(Root1, Root2)
#
# NOTE — cross-species comparability:
#   The maize Tau_Index used N=24 tissues. The brassica Tau_Index uses N=6
#   distinct tissue conditions. Tau values are not numerically comparable
#   between the maize and brassica models. Both correctly capture tissue
#   specificity within their respective species.

exp_df = pd.read_csv(DATA_DIR / "Expression_Features_LogTau_Brapa.csv")
print(f"Expression feature matrix shape: {exp_df.shape}")
print(f"Columns: {exp_df.columns.tolist()}")
display(exp_df.head())

Expression feature matrix shape: (38200, 3)
Columns: ['gene_id', 'Log2_Average', 'Tau_Index']


,gene_id,Log2_Average,Tau_Index
0,Bra000001,0.650350,0.884038
1,Bra000002,1.056030,0.794446
2,Bra000003,4.608238,0.122895
3,Bra000004,2.243178,0.283417
4,Bra000005,5.570712,0.188218


### Expression Data Pre-check

In [18]:
print("=== Expression Data Pre-check ===\n")
print(f"Input dataframe shape: {exp_df.shape}")
print(f"Columns: {exp_df.columns.tolist()}\n")

# --- Hard duplicate check ---
# Duplicates indicate an upstream processing error in the expression script.
duplicates = exp_df['gene_id'].duplicated()
if duplicates.any():
    dup_ids = exp_df.loc[duplicates, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in expression "
        f"input file. Recheck brapa_expression_log2_tau_processing.py output.\n"
        f"Affected IDs: {dup_ids}"
    )
print("✓ No duplicate gene_ids detected\n")

# --- Coverage check against master doublet gene set ---
# 340 master doublet genes are absent from the GSE43245 FPKM file.
# These are real genes in the model pairs with no expression data available.
# They will produce NaN in both Log2_Average and Tau_Index after the merge,
# and every pair containing one of these genes will be dropped at the
# final pair-wise dropout step.
file_genes = set(exp_df['gene_id'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master doublet genes are absent "
          f"from the expression file.")
    print(f"  These genes have no FPKM data in GSE43245, likely due to a "
          f"difference in gene annotation versions between the expression")
    print(f"  dataset and the B. rapa v1.5 subgenome partition.")
    print(f"  NaN will be introduced for both Log2_Average and Tau_Index.")
    print(f"  All pairs containing these genes will be dropped at the "
          f"final pair-wise dropout step.")
    print(f"  Absent gene IDs (first 10): "
          f"{sorted(list(absent_genes))[:10]}"
          f"{'...' if len(absent_genes) > 10 else ''}\n")
else:
    print("✓ All master doublet genes present in expression file\n")

# --- Check Log2_Average for NaNs ---
# NaN in Log2_Average is real missing data requiring pair-wise dropout.
nan_log2_mask = exp_df['Log2_Average'].isna()
if nan_log2_mask.any():
    nan_log2_genes = exp_df.loc[nan_log2_mask, 'gene_id'].tolist()
    print(f"WARNING: {len(nan_log2_genes)} genes have NaN in Log2_Average.")
    print(f"  These genes and their pair partners will be dropped at dropout.")
    print(f"  Affected gene IDs: {nan_log2_genes}\n")
else:
    print("✓ No NaN values in Log2_Average\n")

# --- Check Tau_Index for NaNs separately ---
# NaN in Tau_Index is biologically meaningful: FPKM = 0 across all 8 tissues.
# These NaNs are RETAINED. Tau is undefined, not missing, for these genes.
nan_tau_mask = exp_df['Tau_Index'].isna()
if nan_tau_mask.any():
    print(f"INFO: {nan_tau_mask.sum()} genes have NaN in Tau_Index.")
    print(f"  These genes have FPKM = 0 across all 8 tissues.")
    print(f"  Tau is mathematically undefined for unexpressed genes.")
    print(f"  These NaN values are biologically meaningful and will be "
          f"RETAINED in the final dataset.\n")
else:
    print("✓ No NaN values in Tau_Index\n")

print("=== Pre-check complete. Review warnings before proceeding. ===")

=== Expression Data Pre-check ===

Input dataframe shape: (38200, 3)
Columns: ['gene_id', 'Log2_Average', 'Tau_Index']

✓ No duplicate gene_ids detected

  These genes have no FPKM data in GSE43245, likely due to a difference in gene annotation versions between the expression
  dataset and the B. rapa v1.5 subgenome partition.
  NaN will be introduced for both Log2_Average and Tau_Index.
  All pairs containing these genes will be dropped at the final pair-wise dropout step.
  Absent gene IDs (first 10): ['Bra000104', 'Bra000824', 'Bra001018', 'Bra001108', 'Bra001228', 'Bra001252', 'Bra001309', 'Bra001404', 'Bra001439', 'Bra001465']...

✓ No NaN values in Log2_Average

INFO: 1491 genes have NaN in Tau_Index.
  These genes have FPKM = 0 across all 8 tissues.
  Tau is mathematically undefined for unexpressed genes.
  These NaN values are biologically meaningful and will be RETAINED in the final dataset.

=== Pre-check complete. Review warnings before proceeding. ===


### Expression Data Processing and Merge

In [19]:
print("=== Expression Data Processing and Merge ===\n")

exp_clean = exp_df[['gene_id', 'Log2_Average', 'Tau_Index']]

# --- Merge into lf_mf1_pairs ---
exp_LF = exp_clean.rename(columns={
    'gene_id':      'LF',
    'Log2_Average': 'avg_expression_LF',
    'Tau_Index':    'tau_LF'
})
exp_MF1 = exp_clean.rename(columns={
    'gene_id':      'MF1',
    'Log2_Average': 'avg_expression_MF1',
    'Tau_Index':    'tau_MF1'
})
exp_MF2 = exp_clean.rename(columns={
    'gene_id':      'MF2',
    'Log2_Average': 'avg_expression_MF2',
    'Tau_Index':    'tau_MF2'
})

lf_mf1_pairs = pd.merge(lf_mf1_pairs, exp_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, exp_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, exp_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, exp_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, exp_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, exp_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after expression merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after expression merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after expression merge, expected 2504."

print("✓ Row counts verified after expression merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
# Distinguishes between real missing data (Log2_Average) and
# biologically meaningful NaNs (Tau_Index).
print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        avg_col = f'avg_expression_{sg}'
        tau_col = f'tau_{sg}'

        n_avg = df_pair[avg_col].isna().sum()
        n_tau = df_pair[tau_col].isna().sum()

        if n_avg > 0:
            print(f"  {label} | {avg_col}: {n_avg} NaNs "
                  f"— real missing data, pairs will be dropped at dropout step")
        else:
            print(f"  ✓  {label} | {avg_col}: no NaNs")

        if n_tau > 0:
            print(f"  {label} | {tau_col}: {n_tau} NaNs "
                  f"— biologically meaningful (FPKM = 0), will be RETAINED")
        else:
            print(f"  ✓  {label} | {tau_col}: no NaNs")

=== Expression Data Processing and Merge ===

✓ Row counts verified after expression merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | avg_expression_LF: 72 NaNs — real missing data, pairs will be dropped at dropout step
  lf_mf1 | tau_LF: 111 NaNs — biologically meaningful (FPKM = 0), will be RETAINED
  lf_mf1 | avg_expression_MF1: 87 NaNs — real missing data, pairs will be dropped at dropout step
  lf_mf1 | tau_MF1: 165 NaNs — biologically meaningful (FPKM = 0), will be RETAINED
  lf_mf2 | avg_expression_LF: 61 NaNs — real missing data, pairs will be dropped at dropout step
  lf_mf2 | tau_LF: 93 NaNs — biologically meaningful (FPKM = 0), will be RETAINED
  lf_mf2 | avg_expression_MF2: 109 NaNs — real missing data, pairs will be dropped at dropout step
  lf_mf2 | tau_MF2: 159 NaNs — biologically meaningful (FPKM = 0), will be RETAINED
  mf1_mf2 | avg_expression_MF1: 49 NaNs — real missing data, pairs will be dropped at dropout s

In [20]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after expression merge")

=== Pair-wise Dropout Preview (after expression merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 153 (3.6%)
    Pairs to keep : 4141 (96.4%)
    Columns driving dropout:
      avg_expression_MF1: 87 affected pairs
      avg_expression_LF: 72 affected pairs

  lf_mf2_pairs:
    Total pairs   : 3666
    Pairs to drop : 159 (4.3%)
    Pairs to keep : 3507 (95.7%)
    Columns driving dropout:
      avg_expression_MF2: 109 affected pairs
      avg_expression_LF: 61 affected pairs

  mf1_mf2_pairs:
    Total pairs   : 2504
    Pairs to drop : 121 (4.8%)
    Pairs to keep : 2383 (95.2%)
    Columns driving dropout:
      avg_expression_MF2: 80 affected pairs
      avg_expression_MF1: 49 affected pairs

Note: counts will change as more features are added. Run after e

#### Note: Tau NaNs Retained

NaNs in `tau_LF`, `tau_MF1`, and `tau_MF2` are retained throughout the <br>
preprocessing pipeline. A gene with FPKM = 0 across all 8 tissues has no <br>
defined tissue specificity — Tau is mathematically undefined, not missing. <br>
This is the same treatment applied in the maize preprocessing notebook. <br>

NaNs in `avg_expression_LF`, `avg_expression_MF1`, and `avg_expression_MF2` <br>
are real missing data arising from 340 master doublet genes absent from the <br>
GSE43245 FPKM file. Pairs containing these genes will be removed at the <br>
final pair-wise dropout step. <br>

## GC Content: Gene and Promoter

In [21]:
# Load precomputed GC content features for B. rapa genes.
# Generated by brapa_gc_content_processing.py from primary genomic files:
#   - Genome FASTA : Brapa_sequence_v1.5.fa
#   - Annotation   : Brapa_gene_v1.5.gff
#
# Two features per gene:
#   Gene_CDS_GC  — GC content (%) of the full coding sequence, reconstructed
#                  by concatenating all annotated CDS exons from the genome.
#                  GC content is strand-symmetric so no reverse complementation
#                  is needed.
#   Promoter_GC  — GC content (%) of the 170-bp core promoter window
#                  (-165 to +5 bp relative to the GFF-derived TSS), reverse
#                  complemented for - strand genes. Matches the spatial
#                  definition used for the maize model (Jores et al. 2021).
#
# Known NaN sources in this file:
#   (1) Genes absent from the GFF annotation — receive NaN for both features.
#   (2) Genes whose promoter window falls partly outside a chromosome or
#       scaffold boundary — receive NaN for Promoter_GC only.
#       From pre-analysis: all boundary cases are scaffold-resident genes.
#
# Note on scaffold genes:
#   170 of the 15,903 master doublet genes are on scaffold chromosomes rather
#   than on main chromosomes A01-A10. All 170 have valid AK block assignments
#   and are included. See brapa_gc_content_processing.py for full rationale.

gc_df = pd.read_csv(DATA_DIR / "gc_gene_promoter_output.csv")

print(f"GC content dataframe shape: {gc_df.shape}")
print(f"Columns: {gc_df.columns.tolist()}")
display(gc_df.head())

GC content dataframe shape: (15903, 3)
Columns: ['Gene_ID', 'Gene_CDS_GC', 'Promoter_GC']


,Gene_ID,Gene_CDS_GC,Promoter_GC
0,Bra000001,43.894389,39.411765
1,Bra000003,46.315789,35.294118
2,Bra000005,50.557880,48.823529
3,Bra000007,55.274262,28.235294
4,Bra000008,47.845805,34.117647


### GC Content Pre-check

In [22]:
print("=== GC Content Pre-check ===\n")

# --- Duplicate check ---
duplicates = gc_df['Gene_ID'].duplicated()
if duplicates.any():
    dup_ids = gc_df.loc[duplicates, 'Gene_ID'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate Gene_IDs detected in GC content "
        f"input file. Recheck brapa_gc_content_processing.py output.\n"
        f"Affected IDs: {dup_ids}"
    )
print("✓ No duplicate Gene_IDs detected\n")

# --- Coverage check ---
# Genes absent here will produce NaN for both GC features after the merge.
# Their pairs will be dropped at the final dropout step.
file_genes  = set(gc_df['Gene_ID'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master doublet genes absent from "
          f"GC content file.")
    print(f"  These genes will receive NaN for both Gene_CDS_GC and "
          f"Promoter_GC.")
    print(f"  All pairs containing these genes will be dropped at dropout.")
    print(f"  Absent IDs (first 10): "
          f"{sorted(list(absent_genes))[:10]}"
          f"{'...' if len(absent_genes) > 10 else ''}\n")
else:
    print("✓ All master doublet genes present in GC content file\n")

# --- Value range check ---
# GC content must be in [0, 100]. NaNs are excluded from this check.
for col in ['Gene_CDS_GC', 'Promoter_GC']:
    vals = gc_df[col].dropna()
    if (vals < 0).any() or (vals > 100).any():
        raise ValueError(
            f"CRITICAL: Values outside [0, 100] detected in {col}. "
            f"Review brapa_gc_content_processing.py output."
        )
    print(f"✓ {col}: range {vals.min():.2f}% – {vals.max():.2f}%")

# --- NaN report ---
n_nan_cds  = gc_df['Gene_CDS_GC'].isna().sum()
n_nan_prom = gc_df['Promoter_GC'].isna().sum()

print(f"\nNaN in Gene_CDS_GC : {n_nan_cds}")
print(f"NaN in Promoter_GC : {n_nan_prom}")

if n_nan_prom > n_nan_cds:
    print(f"  {n_nan_prom - n_nan_cds} genes have Promoter_GC NaN but valid "
          f"Gene_CDS_GC — these are boundary truncation cases (scaffold edge).")

print(f"\n=== Pre-check complete ===")

=== GC Content Pre-check ===

✓ No duplicate Gene_IDs detected

✓ All master doublet genes present in GC content file

✓ Gene_CDS_GC: range 36.22% – 64.13%
✓ Promoter_GC: range 1.76% – 70.00%

NaN in Gene_CDS_GC : 0
NaN in Promoter_GC : 0

=== Pre-check complete ===


### GC Content Processing and Merge

In [23]:
print("=== GC Content Processing and Merge ===\n")

gc_clean = gc_df.rename(columns={'Gene_ID': 'gene_id'})

# Build one renamed copy per subgenome.
# gene_id is renamed to the subgenome column name so pd.merge()
# can join on it directly — the same pattern used for histone and expression.
gc_LF = gc_clean.rename(columns={
    'gene_id':     'LF',
    'Gene_CDS_GC': 'gene_cds_gc_LF',
    'Promoter_GC': 'promoter_gc_LF'
})
gc_MF1 = gc_clean.rename(columns={
    'gene_id':     'MF1',
    'Gene_CDS_GC': 'gene_cds_gc_MF1',
    'Promoter_GC': 'promoter_gc_MF1'
})
gc_MF2 = gc_clean.rename(columns={
    'gene_id':     'MF2',
    'Gene_CDS_GC': 'gene_cds_gc_MF2',
    'Promoter_GC': 'promoter_gc_MF2'
})

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, gc_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, gc_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, gc_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, gc_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, gc_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, gc_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after GC merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after GC merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after GC merge, expected 2504."

print("✓ Row counts verified after GC content merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
gc_feature_pairs = [
    ('gene_cds_gc', 'Coding sequence GC'),
    ('promoter_gc', 'Promoter GC'),
]

print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        for feat, feat_label in gc_feature_pairs:
            col = f'{feat}_{sg}'
            n_nan = df_pair[col].isna().sum()
            if n_nan > 0:
                print(f"  {label} | {col}: {n_nan} NaNs "
                      f"— pairs will be dropped at dropout step")
            else:
                print(f"  ✓  {label} | {col}: no NaNs")

=== GC Content Processing and Merge ===

✓ Row counts verified after GC content merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  ✓  lf_mf1 | gene_cds_gc_LF: no NaNs
  ✓  lf_mf1 | promoter_gc_LF: no NaNs
  ✓  lf_mf1 | gene_cds_gc_MF1: no NaNs
  ✓  lf_mf1 | promoter_gc_MF1: no NaNs
  ✓  lf_mf2 | gene_cds_gc_LF: no NaNs
  ✓  lf_mf2 | promoter_gc_LF: no NaNs
  ✓  lf_mf2 | gene_cds_gc_MF2: no NaNs
  ✓  lf_mf2 | promoter_gc_MF2: no NaNs
  ✓  mf1_mf2 | gene_cds_gc_MF1: no NaNs
  ✓  mf1_mf2 | promoter_gc_MF1: no NaNs
  ✓  mf1_mf2 | gene_cds_gc_MF2: no NaNs
  ✓  mf1_mf2 | promoter_gc_MF2: no NaNs


In [24]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after GC content merge")

=== Pair-wise Dropout Preview (after expression merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 153 (3.6%)
    Pairs to keep : 4141 (96.4%)
    Columns driving dropout:
      avg_expression_MF1: 87 affected pairs
      avg_expression_LF: 72 affected pairs

  lf_mf2_pairs:
    Total pairs   : 3666
    Pairs to drop : 159 (4.3%)
    Pairs to keep : 3507 (95.7%)
    Columns driving dropout:
      avg_expression_MF2: 109 affected pairs
      avg_expression_LF: 61 affected pairs

  mf1_mf2_pairs:
    Total pairs   : 2504
    Pairs to drop : 121 (4.8%)
    Pairs to keep : 2383 (95.2%)
    Columns driving dropout:
      avg_expression_MF2: 80 affected pairs
      avg_expression_MF1: 49 affected pairs

Note: counts will change as more features are added. Run after e

## Recombination Rates

In [25]:
# Load recombination rate data for B. rapa genes.
# Source: Zhao et al. (2013) Plant Journal 76:211-222.
# Rates were estimated using MareyMap software, which fits a LOESS spline
# to the relationship between genetic distance (cM) and physical distance
# (bp) along each chromosome, then estimates the local recombination rate
# as the derivative of the fitted curve (cM/Mb).
#
# IMPORTANT — column names in this file are misleading and do not match
# their actual content. Columns are renamed on load to avoid confusion:
#   'phys'    -> 'gene_id'   (contains BraXXXXXX gene identifiers)
#   'default' -> 'phys_pos'  (contains physical midpoint position in bp)
#   'name'    -> 'rec_rate'  (contains recombination rate in cM/Mb)
#   'mkr'     -> 'chrom'     (chromosome, A01-A10)
#   'set','map' are species metadata and are dropped after loading.
#
# NOTE — negative recombination rates:
#   2,731 genes have negative rate values (range: -9.53 to -0.01 cM/Mb).
#   These are a known artifact of LOESS spline smoothing in regions of
#   low marker density or near chromosome boundaries, where the spline
#   derivative can become slightly negative. These values are biologically
#   impossible but are retained in the dataset as-is, consistent with the
#   treatment of the analogous maize recombination data. The presence of
#   negative values is documented in the Methods section.
#
# NOTE — scaffold genes:
#   The recombination rate file covers only genes on main chromosomes
#   A01-A10. The 170 master doublet genes located on scaffold chromosomes
#   are absent from this file and will receive NaN for rec_rate after
#   the merge. Because no genetic map exists for unplaced scaffolds,
#   this data cannot be imputed. These 170 genes and their pair partners
#   will be dropped at the final dropout step.
#
# FILE FORMAT NOTE:
#   Two rows in the source file have a trailing tab (7 fields instead of 6).
#   These are loaded with on_bad_lines='skip' and the two affected rows
#   (Bra000294 and Bra003738) are confirmed to be minor data artifacts
#   with no impact on the doublet gene set.

rec_df = pd.read_csv(
    DATA_DIR / "Brapa_gene_v1.1_gene_sorted_chromosome_GR_New-1.txt",
    sep='\t',
    on_bad_lines='skip'
)

# Rename columns to reflect actual content
rec_df = rec_df.rename(columns={
    'mkr':     'chrom',
    'phys':    'gene_id',
    'default': 'phys_pos',
    'name':    'rec_rate'
})

# Drop species metadata columns — not used downstream
rec_df = rec_df.drop(columns=['set', 'map'])

print(f"Recombination rate dataframe shape: {rec_df.shape}")
print(f"Columns: {rec_df.columns.tolist()}")
display(rec_df.head())

Recombination rate dataframe shape: (39752, 4)
Columns: ['chrom', 'gene_id', 'phys_pos', 'rec_rate']


,chrom,gene_id,phys_pos,rec_rate
0,A01,Bra011024,4366644.5,3.87
1,A01,Bra011025,4358783.5,3.87
2,A01,Bra011026,4356133.0,3.87
3,A01,Bra011027,4353704.5,3.87
4,A01,Bra011028,4351836.0,3.87


### Recombination Rate Pre-check

In [26]:
print("=== Recombination Rate Pre-check ===\n")

# --- Duplicate check ---
duplicates = rec_df['gene_id'].duplicated()
if duplicates.any():
    dup_ids = rec_df.loc[duplicates, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in "
        f"recombination rate file.\nAffected IDs: {dup_ids}"
    )
print("✓ No duplicate gene_ids detected\n")

# --- Chromosome check ---
expected_chroms = {f'A{str(i).zfill(2)}' for i in range(1, 11)}
found_chroms = set(rec_df['chrom'].unique())
if found_chroms != expected_chroms:
    unexpected = found_chroms - expected_chroms
    missing    = expected_chroms - found_chroms
    if unexpected:
        raise ValueError(
            f"CRITICAL: Unexpected chromosomes in recombination file: "
            f"{unexpected}. Scaffold genes should not be present."
        )
    if missing:
        print(f"WARNING: Chromosomes missing from recombination file: {missing}")
else:
    print(f"✓ All 10 main chromosomes present (A01-A10)\n")

# --- Negative rate report ---
n_neg = (rec_df['rec_rate'] < 0).sum()
print(f"INFO: {n_neg} genes have negative recombination rates "
      f"(range: {rec_df['rec_rate'].min():.2f} to "
      f"{rec_df[rec_df['rec_rate'] < 0]['rec_rate'].max():.2f} cM/Mb).")
print(f"  These are LOESS spline artifacts and are retained as-is.")
print(f"  Chromosome breakdown of negative rates:")
neg_by_chrom = rec_df[rec_df['rec_rate'] < 0].groupby('chrom')['rec_rate'].count()
for chrom, count in neg_by_chrom.items():
    print(f"    {chrom}: {count}")
print()

# --- Coverage check against master doublet gene set ---
file_genes  = set(rec_df['gene_id'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"INFO: {len(absent_genes)} master doublet genes are absent "
          f"from the recombination rate file.")
    print(f"  All absent genes are on scaffold chromosomes (A01-A10 only "
          f"are covered).")
    print(f"  No genetic map exists for unplaced scaffolds — this data")
    print(f"  cannot be imputed.")
    print(f"  These genes will receive NaN for rec_rate and their pair")
    print(f"  partners will be dropped at the final dropout step.")
    print(f"  Absent IDs (first 10): "
          f"{sorted(list(absent_genes))[:10]}"
          f"{'...' if len(absent_genes) > 10 else ''}\n")
else:
    print("✓ All master doublet genes present in recombination file\n")

# --- Value range check ---
print(f"rec_rate range: {rec_df['rec_rate'].min():.4f} to "
      f"{rec_df['rec_rate'].max():.4f} cM/Mb")
print(f"rec_rate NaNs: {rec_df['rec_rate'].isna().sum()}")
print(f"\n=== Pre-check complete ===")

=== Recombination Rate Pre-check ===

✓ No duplicate gene_ids detected

✓ All 10 main chromosomes present (A01-A10)

INFO: 2731 genes have negative recombination rates (range: -9.53 to -0.01 cM/Mb).
  These are LOESS spline artifacts and are retained as-is.
  Chromosome breakdown of negative rates:
    A02: 134
    A03: 907
    A04: 219
    A06: 220
    A08: 367
    A10: 884

INFO: 170 master doublet genes are absent from the recombination rate file.
  All absent genes are on scaffold chromosomes (A01-A10 only are covered).
  No genetic map exists for unplaced scaffolds — this data
  cannot be imputed.
  These genes will receive NaN for rec_rate and their pair
  partners will be dropped at the final dropout step.
  Absent IDs (first 10): ['Bra034482', 'Bra034494', 'Bra034495', 'Bra034504', 'Bra034506', 'Bra034515', 'Bra034519', 'Bra034522', 'Bra035332', 'Bra035335']...

rec_rate range: -9.5300 to 47.6400 cM/Mb
rec_rate NaNs: 0

=== Pre-check complete ===


### Recombination Rate Processing and Merge

In [27]:
print("=== Recombination Rate Processing and Merge ===\n")

rec_clean = rec_df[['gene_id', 'rec_rate']]

# Build one renamed copy per subgenome.
rec_LF = rec_clean.rename(columns={
    'gene_id':  'LF',
    'rec_rate': 'rec_rate_LF'
})
rec_MF1 = rec_clean.rename(columns={
    'gene_id':  'MF1',
    'rec_rate': 'rec_rate_MF1'
})
rec_MF2 = rec_clean.rename(columns={
    'gene_id':  'MF2',
    'rec_rate': 'rec_rate_MF2'
})

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, rec_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, rec_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, rec_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, rec_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, rec_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, rec_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after " \
    f"recombination merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after " \
    f"recombination merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after " \
    f"recombination merge, expected 2504."

print("✓ Row counts verified after recombination rate merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        col   = f'rec_rate_{sg}'
        n_nan = df_pair[col].isna().sum()
        if n_nan > 0:
            print(f"  {label} | {col}: {n_nan} NaNs "
                  f"— scaffold genes absent from genetic map, "
                  f"pairs will be dropped at dropout step")
        else:
            print(f"  ✓  {label} | {col}: no NaNs")

=== Recombination Rate Processing and Merge ===

✓ Row counts verified after recombination rate merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | rec_rate_LF: 31 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step
  lf_mf1 | rec_rate_MF1: 51 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step
  lf_mf2 | rec_rate_LF: 39 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step
  lf_mf2 | rec_rate_MF2: 22 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step
  mf1_mf2 | rec_rate_MF1: 52 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step
  mf1_mf2 | rec_rate_MF2: 19 NaNs — scaffold genes absent from genetic map, pairs will be dropped at dropout step


In [28]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after recombination rate merge")

=== Pair-wise Dropout Preview (after recombination rate merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 233 (5.4%)
    Pairs to keep : 4061 (94.6%)
    Columns driving dropout:
      avg_expression_MF1: 87 affected pairs
      avg_expression_LF: 72 affected pairs
      rec_rate_MF1: 51 affected pairs
      rec_rate_LF: 31 affected pairs

  lf_mf2_pairs:
    Total pairs   : 3666
    Pairs to drop : 217 (5.9%)
    Pairs to keep : 3449 (94.1%)
    Columns driving dropout:
      avg_expression_MF2: 109 affected pairs
      avg_expression_LF: 61 affected pairs
      rec_rate_LF: 39 affected pairs
      rec_rate_MF2: 22 affected pairs

  mf1_mf2_pairs:
    Total pairs   : 2504
    Pairs to drop : 188 (7.5%)
    Pairs to keep : 2316 (92.5%)
    Columns driving drop

## TE Data

In [29]:
# === TE Data ===
#
# Three input files provide TE features, each with a different format:
#
# 1. TE TYPE FILE (Brapa_gene_v1.5_genes-1_sorted_TE_type)
#    BEDtools closest output against CLASSIFIED TEs only.
#    One row per gene — no duplicates confirmed.
#    col 3 : gene ID
#    col 6 : TE type of the nearest classified TE
#
# 2. TE DISTANCE FILE (Brapa_gene_v1.5_genes_sorted_DisToTE)
#    BEDtools closest output against ALL TEs, including unclassified.
#    May contain duplicate gene IDs (multiple equidistant TEs).
#    col 3  : gene ID
#    col 10 : distance in bp to the nearest TE
#
# 3. TE DENSITY FILES (upstream and downstream, 2kb, 200 windows each)
#    One row per gene, 201 columns: gene_id + 200 sliding window values.
#    Two separate files for upstream and downstream, unlike the maize
#    model which used a single 400-window file.
#
# NOTE — distance and type describe different TE elements:
#   The distance file includes unclassified TEs (~66% of nearest TEs are
#   'Unknown' family). The type file uses only classified TEs, which may
#   be more distal. TE distance therefore reflects the absolute nearest
#   annotated element; TE type reflects the nearest element with a
#   classified family designation. This is documented here because in
#   the maize model both features were derived from the same TE element.
#   The distinction is retained because using type from the distance file
#   would produce NaN for ~66% of genes, making the feature uninformative.

te_type_df = pd.read_csv(
    DATA_DIR / 'Brapa_gene_v1.5_genes-1_sorted_TE_type',
    sep='\t', header=None
)

te_dist_df = pd.read_csv(
    DATA_DIR / 'Brapa_gene_v1.5_genes_sorted_DisToTE',
    sep='\t', header=None
)

te_up_df = pd.read_csv(
    DATA_DIR / 'Brapa_gene_v1.5_genes-1_sorted_wins_2kbup_TEs_pro-1',
    sep='\t', header=None
)

te_down_df = pd.read_csv(
    DATA_DIR / 'Brapa_gene_v1.5_genes-1_sorted_wins_2kbdown_TEs_pro-1',
    sep='\t', header=None
)

# Assign column names
te_type_df = te_type_df.rename(columns={3: 'gene_id', 6: 'TE_type'})
te_dist_df = te_dist_df.rename(columns={3: 'gene_id', 10: 'te_dist'})

te_up_df.columns   = ['gene_id'] + [f'col{i}' for i in range(1, len(te_up_df.columns))]
te_down_df.columns = ['gene_id'] + [f'col{i}' for i in range(1, len(te_down_df.columns))]

# BEDtools closest -d outputs -1 for all TE fields when no TE is found
# on the scaffold containing the query gene. These are sentinel rows,
# not real distances. Filter them before any downstream processing.
# All five affected genes are on scaffold chromosomes and are absent
# from the master doublet set — this is confirmed by investigation.
n_sentinel = (te_dist_df['te_dist'] == -1).sum()
if n_sentinel > 0:
    print(f"  INFO: {n_sentinel} rows with te_dist == -1 removed "
          f"(BEDtools no-feature sentinel — genes on scaffolds with no "
          f"annotated TEs).")
    te_dist_df = te_dist_df[te_dist_df['te_dist'] != -1].reset_index(drop=True)

print(f"te_type_df shape : {te_type_df.shape}")
print(f"te_dist_df shape : {te_dist_df.shape}")
print(f"te_up_df shape   : {te_up_df.shape}")
print(f"te_down_df shape : {te_down_df.shape}")

  INFO: 5 rows with te_dist == -1 removed (BEDtools no-feature sentinel — genes on scaffolds with no annotated TEs).
te_type_df shape : (40915, 8)
te_dist_df shape : (55882, 11)
te_up_df shape   : (40915, 201)
te_down_df shape : (40915, 201)


### TE Data Pre-check

In [30]:
print("=== TE Data Pre-checks ===\n")
warnings = 0

# --- TE type file ---
print("--- TE Type File ---")

# Hard duplicate check — confirmed zero from upstream analysis
dup_type = te_type_df['gene_id'].duplicated()
if dup_type.any():
    raise ValueError(
        f"CRITICAL: {dup_type.sum()} duplicate gene_ids in TE type file. "
        f"Upstream BEDtools run must be re-examined.\n"
        f"Affected IDs: {te_type_df.loc[dup_type, 'gene_id'].unique().tolist()}"
    )
print("  ✓ No duplicate gene_ids in TE type file")

# Validate all TE types can be mapped by group_dict (defined below)
# 'DNA' is included here — it was absent from the draft dict (1321 genes)
# and has been added as group 2 (broad DNA transposon class)
group_dict = {
    'RC/Helitron': 1,

    'DNA': 2,               # broad class — added; absent from draft dict
    'DNA/CMC-EnSpm': 2,
    'DNA/hAT': 2,
    'DNA/hAT-Ac': 2,
    'DNA/hAT-Tag1': 2,
    'DNA/hAT-Tip100': 2,
    'DNA/MuLE-MuDR': 2,
    'DNA/MULE-MuDR': 2,
    'DNA/PIF-Harbinger': 2,
    'DNA/TcMar-Mariner': 2,
    'DNA/TcMar-Pogo': 2,
    'DNA/TcMar-Stowaway': 2,
    'DNA/TcMar-Tc1': 2,

    'LTR': 3,
    'LTR/Caulimovirus': 3,
    'LTR/Copia': 3,
    'LTR/Gypsy': 3,
    'LTR/Ngaro': 3,
    'LTR/Pao': 3,

    'LINE': 4,
    'LINE/CR1': 4,
    'LINE/I': 4,
    'LINE/L1': 4,
    'LINE/L2': 4,
    'LINE/Tad1': 4,
    'SINE': 4,
    'SINE?': 4,
    'SINE/tRNA': 4,
}

observed_types = set(te_type_df['TE_type'].dropna().unique())
unmapped = observed_types - set(group_dict.keys())
if unmapped:
    raise ValueError(
        f"CRITICAL: {len(unmapped)} TE types in the type file have no "
        f"mapping in group_dict and would silently produce NaN:\n{unmapped}\n"
        f"Add these to group_dict before proceeding."
    )
print(f"  ✓ All {len(observed_types)} TE types in type file are covered "
      f"by group_dict")

# Coverage check
type_genes   = set(te_type_df['gene_id'])
absent_type  = master_genes - type_genes
if absent_type:
    print(f"  WARNING: {len(absent_type)} master genes absent from TE type "
          f"file — will receive NaN for all TEGroup columns.")
    warnings += 1
else:
    print(f"  ✓ All master genes present in TE type file")

# --- TE distance file ---
print("\n--- TE Distance File ---")

# Duplicates are expected and resolved by minimum distance
dup_dist = te_dist_df['gene_id'].duplicated()
n_dup_dist = dup_dist.sum()
if n_dup_dist > 0:
    print(f"  INFO: {n_dup_dist} duplicate gene_ids in TE distance file — "
          f"will be resolved by keeping minimum distance per gene.")
else:
    print(f"  ✓ No duplicate gene_ids in TE distance file")

# After sentinel removal, any remaining negative values would indicate
# a genuine BEDtools formatting problem (e.g. overlap reporting artifacts)
# rather than the known no-feature sentinel. Raise a hard error only for
# those cases so the distinction is clear.
n_neg_dist = (te_dist_df['te_dist'] < 0).sum()
if n_neg_dist > 0:
    neg_rows = te_dist_df[te_dist_df['te_dist'] < 0]
    raise ValueError(
        f"CRITICAL: {n_neg_dist} negative TE distance values remain after "
        f"sentinel removal. These are not BEDtools no-feature sentinels "
        f"(those are exactly -1 and have already been removed). This likely "
        f"indicates a gene-TE overlap reported as a negative distance by "
        f"BEDtools. Investigate these rows before proceeding:\n"
        f"{neg_rows[['gene_id', 'te_dist']].to_string()}"
    )
print(f"  ✓ All TE distance values are non-negative after sentinel removal")

dist_genes  = set(te_dist_df['gene_id'])
absent_dist = master_genes - dist_genes
if absent_dist:
    print(f"  WARNING: {len(absent_dist)} master genes absent from TE "
          f"distance file — will receive NaN for te_dist.")
    warnings += 1
else:
    print(f"  ✓ All master genes present in TE distance file")

# --- TE density files ---
print("\n--- TE Density Files ---")

for df_den, label in [(te_up_df, 'upstream'), (te_down_df, 'downstream')]:
    # Validate 201 columns (1 gene_id + 200 windows)
    if len(df_den.columns) != 201:
        raise ValueError(
            f"CRITICAL: TE density {label} file has {len(df_den.columns)} "
            f"columns, expected 201 (gene_id + 200 windows)."
        )
    print(f"  ✓ {label}: 201 columns verified (gene_id + 200 windows)")

    # Hard duplicate check
    dup_den = df_den['gene_id'].duplicated()
    if dup_den.any():
        raise ValueError(
            f"CRITICAL: {dup_den.sum()} duplicate gene_ids in TE density "
            f"{label} file."
        )
    print(f"  ✓ {label}: no duplicate gene_ids")

    # Density values must be in [0, 1]
    vals = df_den.iloc[:, 1:]
    if ((vals < 0) | (vals > 1)).any().any():
        raise ValueError(
            f"CRITICAL: TE density {label} values outside [0, 1] detected."
        )
    print(f"  ✓ {label}: all density values within [0, 1]")

    # Coverage
    den_genes  = set(df_den['gene_id'])
    absent_den = master_genes - den_genes
    if absent_den:
        print(f"  WARNING: {len(absent_den)} master genes absent from TE "
              f"density {label} file.")
        warnings += 1
    else:
        print(f"  ✓ {label}: all master genes present")

print(f"\n=== Pre-checks complete. Total warnings: {warnings} ===")

=== TE Data Pre-checks ===

--- TE Type File ---
  ✓ No duplicate gene_ids in TE type file
  ✓ All 29 TE types in type file are covered by group_dict
  ✓ All master genes present in TE type file

--- TE Distance File ---
  INFO: 14974 duplicate gene_ids in TE distance file — will be resolved by keeping minimum distance per gene.
  ✓ All TE distance values are non-negative after sentinel removal
  ✓ All master genes present in TE distance file

--- TE Density Files ---
  ✓ upstream: 201 columns verified (gene_id + 200 windows)
  ✓ upstream: no duplicate gene_ids
  ✓ upstream: all density values within [0, 1]
  ✓ upstream: all master genes present
  ✓ downstream: 201 columns verified (gene_id + 200 windows)
  ✓ downstream: no duplicate gene_ids
  ✓ downstream: all density values within [0, 1]
  ✓ downstream: all master genes present

=== Pre-checks complete. Total warnings: 0 ===


### TE Processing and Merge

In [31]:
print("=== TE Processing and Merge ===\n")

# --- TE Type: map to groups and one-hot encode ---
#
# TE types are mapped to four broad functional groups matching the maize model:
#   Group 1: RC/Helitron (rolling-circle transposons)
#   Group 2: DNA transposons (TIR elements and related)
#   Group 3: LTR retrotransposons (including solo LTRs)
#   Group 4: Non-LTR retrotransposons (LINE and SINE elements)
#
# One-hot encoding produces four binary columns per gene.
# All four columns are guaranteed present even if a group is absent
# in the data, ensuring consistent column structure across datasets.

te_type_df['TEGroup'] = te_type_df['TE_type'].map(group_dict)

unmapped_after = te_type_df['TEGroup'].isna().sum()
if unmapped_after > 0:
    print(f"WARNING: {unmapped_after} genes have unmapped TE types after "
          f"applying group_dict. These will have NaN for all TEGroup columns.")
else:
    print(f"✓ All TE types successfully mapped to groups")

# One-hot encode using pd.get_dummies (consistent with maize notebook)
te_ohe = pd.get_dummies(te_type_df['TEGroup'], prefix='TEGroup').astype(float)

# Guarantee all four group columns present
for i in range(1, 5):
    col = f'TEGroup_{i}'
    if col not in te_ohe.columns:
        te_ohe[col] = 0.0
        print(f"  INFO: {col} absent from data — column added with 0.0")
te_ohe = te_ohe[[f'TEGroup_{i}' for i in range(1, 5)]]

te_type_clean = pd.concat(
    [te_type_df[['gene_id']], te_ohe], axis=1
)
print(f"TE type feature dataframe shape: {te_type_clean.shape}")

# --- TE Distance: resolve duplicates by minimum distance ---
te_dist_min = (
    te_dist_df
    .groupby('gene_id', as_index=False)['te_dist']
    .min()
)
print(f"TE distance dataframe shape after min-distance deduplication: "
      f"{te_dist_min.shape}")

# --- TE Density: compute summary statistics ---
# Upstream: mean and max across 200 windows
# Downstream: mean and max across 200 windows
# Window columns are col1 through col200 in both files.
window_cols = [f'col{i}' for i in range(1, 201)]

te_up_df['TEdenseAvgUp']  = te_up_df[window_cols].mean(axis=1)
te_up_df['TEdenseMaxUp']  = te_up_df[window_cols].max(axis=1)
te_down_df['TEdenseAvgDown'] = te_down_df[window_cols].mean(axis=1)
te_down_df['TEdenseMaxDown'] = te_down_df[window_cols].max(axis=1)

te_density_clean = te_up_df[['gene_id', 'TEdenseAvgUp', 'TEdenseMaxUp']].merge(
    te_down_df[['gene_id', 'TEdenseAvgDown', 'TEdenseMaxDown']],
    on='gene_id', how='outer'
)
print(f"TE density feature dataframe shape: {te_density_clean.shape}")

# --- Build per-subgenome merge frames ---
def make_te_frames(sg):
    """Build the three TE feature dataframes for one subgenome label."""
    dist = te_dist_min.rename(columns={
        'gene_id': sg,
        'te_dist': f'te_dist_{sg}'
    })
    ttype = te_type_clean.rename(columns={
        'gene_id':    sg,
        'TEGroup_1':  f'TEGroup_1_{sg}',
        'TEGroup_2':  f'TEGroup_2_{sg}',
        'TEGroup_3':  f'TEGroup_3_{sg}',
        'TEGroup_4':  f'TEGroup_4_{sg}',
    })
    dens = te_density_clean.rename(columns={
        'gene_id':        sg,
        'TEdenseAvgUp':   f'TEdenseAvgUp_{sg}',
        'TEdenseMaxUp':   f'TEdenseMaxUp_{sg}',
        'TEdenseAvgDown': f'TEdenseAvgDown_{sg}',
        'TEdenseMaxDown': f'TEdenseMaxDown_{sg}',
    })
    return dist, ttype, dens

te_dist_LF,  te_type_LF,  te_dens_LF  = make_te_frames('LF')
te_dist_MF1, te_type_MF1, te_dens_MF1 = make_te_frames('MF1')
te_dist_MF2, te_type_MF2, te_dens_MF2 = make_te_frames('MF2')

# --- Merge into lf_mf1_pairs ---
for frame in [te_dist_LF, te_type_LF, te_dens_LF,
              te_dist_MF1, te_type_MF1, te_dens_MF1]:
    key = 'LF' if 'LF' in frame.columns else 'MF1'
    lf_mf1_pairs = pd.merge(lf_mf1_pairs, frame, on=key, how='left')

# --- Merge into lf_mf2_pairs ---
for frame in [te_dist_LF, te_type_LF, te_dens_LF,
              te_dist_MF2, te_type_MF2, te_dens_MF2]:
    key = 'LF' if 'LF' in frame.columns else 'MF2'
    lf_mf2_pairs = pd.merge(lf_mf2_pairs, frame, on=key, how='left')

# --- Merge into mf1_mf2_pairs ---
for frame in [te_dist_MF1, te_type_MF1, te_dens_MF1,
              te_dist_MF2, te_type_MF2, te_dens_MF2]:
    key = 'MF1' if 'MF1' in frame.columns else 'MF2'
    mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, frame, on=key, how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after TE merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after TE merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after TE merge, expected 2504."

print("\n✓ Row counts verified after TE merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
te_feature_groups = [
    ('te_dist',        'TE distance'),
    ('TEGroup_1',      'TE group 1 (Helitron)'),
    ('TEGroup_2',      'TE group 2 (DNA)'),
    ('TEGroup_3',      'TE group 3 (LTR)'),
    ('TEGroup_4',      'TE group 4 (LINE/SINE)'),
    ('TEdenseAvgUp',   'TE density avg upstream'),
    ('TEdenseMaxUp',   'TE density max upstream'),
    ('TEdenseAvgDown', 'TE density avg downstream'),
    ('TEdenseMaxDown', 'TE density max downstream'),
]

print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        for feat, feat_label in te_feature_groups:
            col = f'{feat}_{sg}'
            n_nan = df_pair[col].isna().sum()
            if n_nan > 0:
                print(f"  {label} | {col}: {n_nan} NaNs "
                      f"— pairs will be dropped at dropout step")
            else:
                print(f"  ✓  {label} | {col}: no NaNs")

=== TE Processing and Merge ===

✓ All TE types successfully mapped to groups
TE type feature dataframe shape: (40915, 5)
TE distance dataframe shape after min-distance deduplication: (40908, 2)
TE density feature dataframe shape: (40915, 5)

✓ Row counts verified after TE merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  ✓  lf_mf1 | te_dist_LF: no NaNs
  ✓  lf_mf1 | TEGroup_1_LF: no NaNs
  ✓  lf_mf1 | TEGroup_2_LF: no NaNs
  ✓  lf_mf1 | TEGroup_3_LF: no NaNs
  ✓  lf_mf1 | TEGroup_4_LF: no NaNs
  ✓  lf_mf1 | TEdenseAvgUp_LF: no NaNs
  ✓  lf_mf1 | TEdenseMaxUp_LF: no NaNs
  ✓  lf_mf1 | TEdenseAvgDown_LF: no NaNs
  ✓  lf_mf1 | TEdenseMaxDown_LF: no NaNs
  ✓  lf_mf1 | te_dist_MF1: no NaNs
  ✓  lf_mf1 | TEGroup_1_MF1: no NaNs
  ✓  lf_mf1 | TEGroup_2_MF1: no NaNs
  ✓  lf_mf1 | TEGroup_3_MF1: no NaNs
  ✓  lf_mf1 | TEGroup_4_MF1: no NaNs
  ✓  lf_mf1 | TEdenseAvgUp_MF1: no NaNs
  ✓  lf_mf1 | TEdenseMaxUp_MF1: no NaNs
  ✓  lf_mf1 | TEdenseAvgDown_

In [32]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after TE merge")

=== Pair-wise Dropout Preview (after TE merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 233 (5.4%)
    Pairs to keep : 4061 (94.6%)
    Columns driving dropout:
      avg_expression_MF1: 87 affected pairs
      avg_expression_LF: 72 affected pairs
      rec_rate_MF1: 51 affected pairs
      rec_rate_LF: 31 affected pairs

  lf_mf2_pairs:
    Total pairs   : 3666
    Pairs to drop : 217 (5.9%)
    Pairs to keep : 3449 (94.1%)
    Columns driving dropout:
      avg_expression_MF2: 109 affected pairs
      avg_expression_LF: 61 affected pairs
      rec_rate_LF: 39 affected pairs
      rec_rate_MF2: 22 affected pairs

  mf1_mf2_pairs:
    Total pairs   : 2504
    Pairs to drop : 188 (7.5%)
    Pairs to keep : 2316 (92.5%)
    Columns driving dropout:
      avg_e

## Methylation Data

In [33]:
# === DNA Methylation Data ===
#
# Methylation features are derived from bisulfite sequencing data mapped to
# the B. rapa v1.5 genome. Three cytosine methylation contexts are measured:
#   CG  — symmetric methylation, associated with gene body and TE silencing
#   CHG — semi-symmetric methylation, associated with TE silencing
#   CHH — asymmetric methylation, associated with small RNA-directed
#          silencing (RdDM), particularly relevant to subgenome dominance
#          via the 24-nt small RNA pathway described in Cheng et al. (2016)
#
# For each context, methylation is quantified across three genomic regions:
#   2kbup   — 2 kb upstream of the gene (promoter-proximal)
#   2kbdown — 2 kb downstream of the gene
#   body    — gene body
#
# Two summary statistics per context × region:
#   _ave : mean methylation level across all cytosines in the region
#          (column 3 of the file, bounded [0,1])
#   _max : maximum methylation level across any single cytosine window
#          (column 1 of the file, bounded [0,1])
#
# IMPORTANT — body max exclusion:
#   Body max methylation is deliberately excluded from the feature set,
#   consistent with the maize model. Only upstream and downstream max
#   methylation are used. This yields 9 average features and 6 max
#   features per gene (15 total per subgenome).
#
# IMPORTANT — column selection for _ave files:
#   Each _ave file contains 4 columns:
#     col 0: gene_id
#     col 1: sum of methylation values across windows (unbounded, NOT used)
#     col 2: number of windows with coverage
#     col 3: true per-cytosine average = col1 / col2 (bounded [0,1], USED)
#   The previous draft preprocessing notebook did not explicitly select
#   column 3 and may have inadvertently used column 1 (the raw sum).
#   This notebook corrects that by explicitly selecting column 3.

methyl_ave_files = [
    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CG_ave',   'CG_up_ave'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CG_ave', 'CG_down_ave'),
    ('Brapa_gene_v1.5_genes_body_40wins_methylation_CG_ave',    'CG_body_ave'),

    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CHG_ave',   'CHG_up_ave'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CHG_ave', 'CHG_down_ave'),
    ('Brapa_gene_v1.5_genes_body_40wins_methylation_CHG_ave',    'CHG_body_ave'),

    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CHH_ave',   'CHH_up_ave'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CHH_ave', 'CHH_down_ave'),
    ('Brapa_gene_v1.5_genes_body_40wins_methylation_CHH_ave',    'CHH_body_ave'),
]

# Body max is excluded — only upstream and downstream max are loaded
methyl_max_files = [
    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CG_max',   'CG_up_max'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CG_max', 'CG_down_max'),

    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CHG_max',   'CHG_up_max'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CHG_max', 'CHG_down_max'),

    ('Brapa_gene_v1.5_genes_2kbup_40wins_methylation_CHH_max',   'CHH_up_max'),
    ('Brapa_gene_v1.5_genes_2kbdown_40wins_methylation_CHH_max', 'CHH_down_max'),
]

# Body max files exist but are deliberately not loaded
methyl_body_max_files_excluded = [
    'Brapa_gene_v1.5_genes_body_40wins_methylation_CG_max',
    'Brapa_gene_v1.5_genes_body_40wins_methylation_CHG_max',
    'Brapa_gene_v1.5_genes_body_40wins_methylation_CHH_max',
]
print(f"Average methylation files : {len(methyl_ave_files)} "
      f"(3 contexts × 3 regions)")
print(f"Max methylation files     : {len(methyl_max_files)} "
      f"(3 contexts × 2 regions — body max excluded)")
print(f"Body max files excluded   : {methyl_body_max_files_excluded}")

Average methylation files : 9 (3 contexts × 3 regions)
Max methylation files     : 6 (3 contexts × 2 regions — body max excluded)
Body max files excluded   : ['Brapa_gene_v1.5_genes_body_40wins_methylation_CG_max', 'Brapa_gene_v1.5_genes_body_40wins_methylation_CHG_max', 'Brapa_gene_v1.5_genes_body_40wins_methylation_CHH_max']


### Methylation Data Pre-check

In [34]:
print("=== Methylation Data Pre-check ===\n")
warnings = 0

for file_list, stat_type, col_idx in [
    (methyl_ave_files, 'average', 3),
    (methyl_max_files, 'max',     1),
]:
    print(f"--- {stat_type.capitalize()} methylation files (using column {col_idx}) ---")
    for file_path, col_name in file_list:
        df_tmp = pd.read_csv(DATA_DIR / file_path, sep='\t', header=None)

        gene_ids = df_tmp[0]

        # Hard duplicate check
        if gene_ids.duplicated().any():
            dup_ids = gene_ids[gene_ids.duplicated()].unique().tolist()
            raise ValueError(
                f"CRITICAL: {len(dup_ids)} duplicate gene_ids in {col_name}. "
                f"Upstream processing error.\nAffected IDs: {dup_ids}"
            )

        # Value range check on the column we will use
        vals = df_tmp[col_idx].dropna()
        if (vals < 0).any() or (vals > 1).any():
            raise ValueError(
                f"CRITICAL: Values outside [0,1] detected in column "
                f"{col_idx} of {col_name}.\n"
                f"Range: {vals.min():.4f} to {vals.max():.4f}\n"
                f"Confirm column selection is correct."
            )

        # Coverage check
        file_genes  = set(gene_ids)
        absent      = master_genes - file_genes
        if absent:
            print(f"  WARNING: {col_name}: {len(absent)} master genes absent "
                  f"— will receive NaN, pairs dropped at dropout step.")
            warnings += 1
        else:
            print(f"  ✓  {col_name}: all master genes present, "
                  f"values in [0,1]")
    print()

# Confirm body max files are NOT accidentally loaded
print("--- Body max exclusion check ---")
for fname in methyl_body_max_files_excluded:
    if (DATA_DIR / fname).exists():
        print(f"  ✓  {fname}: exists on disk but is correctly NOT loaded.")
    else:
        print(f"  INFO: {fname}: not found on disk.")

print(f"\n=== Pre-check complete. Total warnings: {warnings} ===")

=== Methylation Data Pre-check ===

--- Average methylation files (using column 3) ---

--- Max methylation files (using column 1) ---

--- Body max exclusion check ---
  ✓  Brapa_gene_v1.5_genes_body_40wins_methylation_CG_max: exists on disk but is correctly NOT loaded.
  ✓  Brapa_gene_v1.5_genes_body_40wins_methylation_CHG_max: exists on disk but is correctly NOT loaded.
  ✓  Brapa_gene_v1.5_genes_body_40wins_methylation_CHH_max: exists on disk but is correctly NOT loaded.

=== Pre-check complete. Total warnings: 15 ===


### Methylation Data Processing and Merge

In [35]:
print("=== Methylation Processing and Merge ===\n")

# --- Build single wide methylation dataframe indexed by gene_id ---
# Average files: column 3 is the true per-cytosine average (col1/col2),
# bounded [0,1]. Column 1 (raw sum) is NOT used.
# Max files: column 1 is the max value, bounded [0,1].

methyl_frames = []

for file_path, col_name in methyl_ave_files:
    df_tmp = pd.read_csv(DATA_DIR / file_path, sep='\t', header=None)
    df_tmp = df_tmp[[0, 3]].rename(columns={0: 'gene_id', 3: col_name})
    methyl_frames.append(df_tmp.set_index('gene_id'))

for file_path, col_name in methyl_max_files:
    df_tmp = pd.read_csv(DATA_DIR / file_path, sep='\t', header=None)
    df_tmp = df_tmp[[0, 1]].rename(columns={0: 'gene_id', 1: col_name})
    methyl_frames.append(df_tmp.set_index('gene_id'))

methyl_wide = pd.concat(methyl_frames, axis=1).reset_index()
methyl_wide = methyl_wide.rename(columns={'index': 'gene_id'})

expected_cols = (
    [col for _, col in methyl_ave_files] +
    [col for _, col in methyl_max_files]
)
assert set(expected_cols) == set(methyl_wide.columns) - {'gene_id'}, \
    "CRITICAL: Methylation wide dataframe columns do not match expected set."

# Confirm body max columns are absent
body_max_cols_present = [c for c in methyl_wide.columns if 'body' in c and 'max' in c]
assert len(body_max_cols_present) == 0, \
    f"CRITICAL: Body max columns present in methylation dataframe: " \
    f"{body_max_cols_present}"

print(f"Methylation wide dataframe shape: {methyl_wide.shape}")
print(f"  {len(methyl_ave_files)} average features + "
      f"{len(methyl_max_files)} max features = "
      f"{len(methyl_ave_files) + len(methyl_max_files)} total")
print(f"  ✓ Body max columns confirmed absent")

# --- Build per-subgenome merge frames and merge ---
all_methyl_cols = [col for _, col in methyl_ave_files + methyl_max_files]

methyl_LF  = methyl_wide.rename(
    columns={'gene_id': 'LF',  **{c: f'{c}_LF'  for c in all_methyl_cols}}
)
methyl_MF1 = methyl_wide.rename(
    columns={'gene_id': 'MF1', **{c: f'{c}_MF1' for c in all_methyl_cols}}
)
methyl_MF2 = methyl_wide.rename(
    columns={'gene_id': 'MF2', **{c: f'{c}_MF2' for c in all_methyl_cols}}
)

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, methyl_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, methyl_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, methyl_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, methyl_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, methyl_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, methyl_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after methylation merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after methylation merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after methylation merge, expected 2504."

print("\n✓ Row counts verified after methylation merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        for col in all_methyl_cols:
            full_col = f'{col}_{sg}'
            n_nan = df_pair[full_col].isna().sum()
            if n_nan > 0:
                print(f"  {label} | {full_col}: {n_nan} NaNs "
                      f"— pairs will be dropped at dropout step")
            else:
                print(f"  ✓  {label} | {full_col}: no NaNs")

=== Methylation Processing and Merge ===

Methylation wide dataframe shape: (40140, 16)
  9 average features + 6 max features = 15 total
  ✓ Body max columns confirmed absent

✓ Row counts verified after methylation merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | CG_up_ave_LF: 421 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CG_down_ave_LF: 326 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CG_body_ave_LF: 301 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHG_up_ave_LF: 426 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHG_down_ave_LF: 310 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHG_body_ave_LF: 282 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHH_up_ave_LF: 310 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHH_down_ave_LF: 227 NaNs — pairs will be dropped at dropout step
  lf_mf1 | CHH_body_ave_LF: 220 NaNs — pairs will be dropped at dropout step
  lf

In [36]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after methylation merge")

=== Pair-wise Dropout Preview (after methylation merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 1930 (44.9%)
    Pairs to keep : 2364 (55.1%)
    Columns driving dropout:
      CHG_up_ave_MF1: 463 affected pairs
      CHG_up_max_MF1: 463 affected pairs
      CG_up_max_MF1: 456 affected pairs
      CG_up_ave_MF1: 456 affected pairs
      CHG_up_ave_LF: 426 affected pairs
      CHG_up_max_LF: 426 affected pairs
      CG_up_ave_LF: 421 affected pairs
      CG_up_max_LF: 421 affected pairs
      CG_down_ave_MF1: 405 affected pairs
      CG_down_max_MF1: 405 affected pairs
      CHG_down_ave_MF1: 355 affected pairs
      CHG_down_max_MF1: 355 affected pairs
      CHH_up_max_MF1: 351 affected pairs
      CHH_up_ave_MF1: 351 affected pairs
      CG_body_ave_MF1: 3

## ACR Data

In [37]:
# Load precomputed ACR features for B. rapa genes.
# Generated by assign_ACR_features_brapa.py, which processes the combined
# 8-library ATAC-seq peak file (ACRs_8_libraries_combined.bed) against
# the B. rapa v1.5 gene annotation using the same strand-aware algorithm
# as the maize ACR preprocessing script.
#
# Three features per gene:
#   summit_fold_enrichment : fold enrichment at the summit of the most
#                            relevant ACR (highest among overlapping ACRs,
#                            or the single closest flanking ACR if none overlap)
#   upstream_distance      : distance in bp to the nearest upstream ACR summit
#   downstream_distance    : distance in bp to the nearest downstream ACR summit
#
# NOTE — fold enrichment distribution:
#   The ACR BED contains a small number of peaks with extreme fold enrichment
#   values (max: 9,195). These are confirmed real peaks with high pileup
#   (not noise artifacts) in regions of near-zero local background. Only
#   26 of 15,903 master doublet genes are within 5 kb of any peak with
#   FE > 100, and only 1 gene is within 5 kb of a peak with FE > 1000.
#   Values are retained as-is and documented in the Methods section.
#
# NOTE — NaN values:
#   NaN in upstream_distance or downstream_distance for a main-chromosome
#   gene indicates no ACR was detected in that direction — typically a gene
#   near a chromosome end. These NaNs are biologically meaningful and are
#   exempt from dropout (see NAN_EXEMPT_COLS in the utility block).
#
#   NaN in summit_fold_enrichment occurs only when both upstream and
#   downstream distances are also NaN (no ACRs anywhere on the chromosome).
#   This happens only for scaffold-resident genes, which are already
#   excluded from the model by missing recombination rate data.
#   summit_fold_enrichment NaNs are therefore NOT exempt from dropout.

acr_df = pd.read_csv(
    '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/'
    'input_ACR/Brapa_ACR_features.tsv',
    sep='\t'
)

print(f"ACR feature dataframe shape: {acr_df.shape}")
print(f"Columns: {acr_df.columns.tolist()}")
display(acr_df.head())

ACR feature dataframe shape: (39609, 4)
Columns: ['gene_id', 'summit_fold_enrichment', 'upstream_distance', 'downstream_distance']


,gene_id,summit_fold_enrichment,upstream_distance,downstream_distance
0,Bra000001,4.91051,48242.0,8032.0
1,Bra000002,4.91051,4748.0,51630.0
2,Bra000003,6.31351,6989.0,7134.0
3,Bra000004,8.34840,11377.0,3173.0
4,Bra000005,8.34840,282.0,12600.0


### ACR Data Pre-check

In [38]:
print("=== ACR Data Pre-check ===\n")
warnings = 0

# --- Duplicate check ---
dups = acr_df['gene_id'].duplicated()
if dups.any():
    raise ValueError(
        f"CRITICAL: {dups.sum()} duplicate gene_ids in ACR feature file. "
        f"Recheck assign_ACR_features_brapa.py output.\n"
        f"Affected IDs: {acr_df.loc[dups, 'gene_id'].unique().tolist()}"
    )
print("✓ No duplicate gene_ids detected\n")

# --- Coverage check ---
# Genes absent from the ACR file altogether (distinct from NaN within the file)
file_genes   = set(acr_df['gene_id'])
absent_genes = master_genes - file_genes
if absent_genes:
    print(f"WARNING: {len(absent_genes)} master genes entirely absent from "
          f"ACR feature file.")
    print(f"  These genes will receive NaN for all three ACR features.")
    print(f"  Absent IDs (first 10): "
          f"{sorted(list(absent_genes))[:10]}"
          f"{'...' if len(absent_genes) > 10 else ''}\n")
    warnings += 1
else:
    print("✓ All master genes present in ACR feature file\n")

# --- NaN report ---
# NaNs here are expected and exempt — report counts for transparency
for col in ['summit_fold_enrichment', 'upstream_distance', 'downstream_distance']:
    n_nan = acr_df[col].isna().sum()
    print(f"  NaN in {col}: {n_nan} "
          f"({'exempt — biologically meaningful' if n_nan > 0 else 'none'})")

print()

# --- Value sanity check ---
# fold_enrichment must be positive; distances must be non-negative
fe_neg  = (acr_df['summit_fold_enrichment'].dropna() <= 0).sum()
ud_neg  = (acr_df['upstream_distance'].dropna() < 0).sum()
dd_neg  = (acr_df['downstream_distance'].dropna() < 0).sum()
if fe_neg > 0:
    raise ValueError(
        f"CRITICAL: {fe_neg} genes have non-positive summit_fold_enrichment. "
        f"Review assign_ACR_features_brapa.py output."
    )
if ud_neg > 0 or dd_neg > 0:
    raise ValueError(
        f"CRITICAL: Negative distance values detected "
        f"(upstream: {ud_neg}, downstream: {dd_neg}). "
        f"Review assign_ACR_features_brapa.py output."
    )
print("✓ All non-NaN fold enrichment values are positive")
print("✓ All non-NaN distance values are non-negative")
print(f"\n  summit_fold_enrichment range: "
      f"{acr_df['summit_fold_enrichment'].min():.2f} to "
      f"{acr_df['summit_fold_enrichment'].max():.2f}")
print(f"  upstream_distance range    : "
      f"{acr_df['upstream_distance'].min():.0f} to "
      f"{acr_df['upstream_distance'].max():.0f} bp")
print(f"  downstream_distance range  : "
      f"{acr_df['downstream_distance'].min():.0f} to "
      f"{acr_df['downstream_distance'].max():.0f} bp")

print(f"\n=== Pre-check complete. Total warnings: {warnings} ===")

=== ACR Data Pre-check ===

✓ No duplicate gene_ids detected

  These genes will receive NaN for all three ACR features.
  Absent IDs (first 10): ['Bra034482', 'Bra034494', 'Bra034495', 'Bra034504', 'Bra034506', 'Bra034515', 'Bra034519', 'Bra034522', 'Bra035332', 'Bra035335']...

  NaN in summit_fold_enrichment: 0 (none)
  NaN in upstream_distance: 25 (exempt — biologically meaningful)
  NaN in downstream_distance: 25 (exempt — biologically meaningful)

✓ All non-NaN fold enrichment values are positive
✓ All non-NaN distance values are non-negative

  summit_fold_enrichment range: 2.97 to 9195.30
  upstream_distance range    : 0 to 283492 bp
  downstream_distance range  : 1 to 266274 bp

=== Pre-check complete. Total warnings: 1 ===


### ACR Data Processing and Merge

In [39]:
print("=== ACR Processing and Merge ===\n")

acr_clean = acr_df[['gene_id', 'summit_fold_enrichment',
                     'upstream_distance', 'downstream_distance']]

acr_LF = acr_clean.rename(columns={
    'gene_id':               'LF',
    'summit_fold_enrichment': 'summit_fold_enrichment_LF',
    'upstream_distance':      'upstream_distance_LF',
    'downstream_distance':    'downstream_distance_LF',
})
acr_MF1 = acr_clean.rename(columns={
    'gene_id':               'MF1',
    'summit_fold_enrichment': 'summit_fold_enrichment_MF1',
    'upstream_distance':      'upstream_distance_MF1',
    'downstream_distance':    'downstream_distance_MF1',
})
acr_MF2 = acr_clean.rename(columns={
    'gene_id':               'MF2',
    'summit_fold_enrichment': 'summit_fold_enrichment_MF2',
    'upstream_distance':      'upstream_distance_MF2',
    'downstream_distance':    'downstream_distance_MF2',
})

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, acr_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, acr_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, acr_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, acr_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, acr_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, acr_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after ACR merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after ACR merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after ACR merge, expected 2504."

print("✓ Row counts verified after ACR merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
# upstream_distance and downstream_distance NaNs are EXEMPT (chromosome ends).
# summit_fold_enrichment NaNs are NOT exempt — they occur only when both
# distance columns are also NaN (scaffold genes), which are already dropped
# by missing recombination rate data.
acr_feature_cols = ['summit_fold_enrichment', 'upstream_distance', 'downstream_distance']

print("\n--- NaN Report ---")
exempt_acr = {'upstream_distance', 'downstream_distance'}

for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        for feat in acr_feature_cols:
            col   = f'{feat}_{sg}'
            n_nan = df_pair[col].isna().sum()
            if n_nan > 0:
                if feat in exempt_acr:
                    print(f"  {label} | {col}: {n_nan} NaNs "
                          f"(exempt — chromosome end, no ACR in this direction)")
                else:
                    print(f"  {label} | {col}: {n_nan} NaNs "
                          f"(NOT exempt — scaffold genes, will be dropped at dropout)")
            else:
                print(f"  ✓  {label} | {col}: no NaNs")

=== ACR Processing and Merge ===

✓ Row counts verified after ACR merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | summit_fold_enrichment_LF: 31 NaNs (NOT exempt — scaffold genes, will be dropped at dropout)
  lf_mf1 | upstream_distance_LF: 32 NaNs (exempt — chromosome end, no ACR in this direction)
  lf_mf1 | downstream_distance_LF: 34 NaNs (exempt — chromosome end, no ACR in this direction)
  lf_mf1 | summit_fold_enrichment_MF1: 51 NaNs (NOT exempt — scaffold genes, will be dropped at dropout)
  lf_mf1 | upstream_distance_MF1: 55 NaNs (exempt — chromosome end, no ACR in this direction)
  lf_mf1 | downstream_distance_MF1: 51 NaNs (exempt — chromosome end, no ACR in this direction)
  lf_mf2 | summit_fold_enrichment_LF: 39 NaNs (NOT exempt — scaffold genes, will be dropped at dropout)
  lf_mf2 | upstream_distance_LF: 40 NaNs (exempt — chromosome end, no ACR in this direction)
  lf_mf2 | downstream_distance_LF: 42 NaNs (exempt — c

In [40]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after ACR merge")

=== Pair-wise Dropout Preview (after ACR merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 1930 (44.9%)
    Pairs to keep : 2364 (55.1%)
    Columns driving dropout:
      CHG_up_max_MF1: 463 affected pairs
      CHG_up_ave_MF1: 463 affected pairs
      CG_up_max_MF1: 456 affected pairs
      CG_up_ave_MF1: 456 affected pairs
      CHG_up_ave_LF: 426 affected pairs
      CHG_up_max_LF: 426 affected pairs
      CG_up_max_LF: 421 affected pairs
      CG_up_ave_LF: 421 affected pairs
      CG_down_max_MF1: 405 affected pairs
      CG_down_ave_MF1: 405 affected pairs
      CHG_down_max_MF1: 355 affected pairs
      CHG_down_ave_MF1: 355 affected pairs
      CHH_up_max_MF1: 351 affected pairs
      CHH_up_ave_MF1: 351 affected pairs
      CG_body_ave_MF1: 344 affec

## Evolutionary Metrics

In [41]:
# Load Ka/Ks/omega evolutionary rate data for B. rapa genes.
# Source: Zhao et al. (2013) Plant Journal 76:211-222.
# Values were computed using the yn00 module in PAML, comparing each
# B. rapa gene against its syntenic Arabidopsis thaliana ortholog.
#
# IMPORTANT — per-gene vs per-pair interpretation:
#   In the maize model, Ka/Ks/omega are pair-specific: computed between
#   the two homeologous copies directly. In brassica, these values are
#   gene-specific: each B. rapa gene is compared to its Arabidopsis
#   ortholog independently. This means the same LF gene's Ka/Ks/omega
#   values appear identically in both the LF-MF1 and LF-MF2 pairwise
#   dataframes. This is not an error — it reflects how the data was
#   generated and is documented here for interpretive clarity.
#
# Column mapping (raw file has two seq. columns — pandas auto-renames):
#   seq.   : B. rapa gene ID (Bra identifier)
#   seq..1 : Arabidopsis thaliana ortholog ID (not used downstream)
#   dN     : non-synonymous substitution rate (Ka)
#   dS     : synonymous substitution rate (Ks)
#   omega  : dN/dS ratio (selection pressure metric)
#   S, N, t, kappa, +-, SE, +-.1, SE.1 : PAML intermediate statistics,
#            not used as model features
#
# Coverage: 7 master doublet genes are absent from this file.
# These receive NaN for all three features and their pairs are dropped
# at the final dropout step.

evol_df = pd.read_csv(
    '/orange/meixiazhao/meixiazhao/Brassica/KaKs/Brapa_Arabidopsis_KaKs_All',
    sep='\t'
)

print(f"Raw Ka/Ks dataframe shape: {evol_df.shape}")
print(f"Columns: {evol_df.columns.tolist()}")
display(evol_df.head())

Raw Ka/Ks dataframe shape: (23718, 13)
Columns: ['seq.', 'seq..1', 'S', 'N', 't', 'kappa', 'omega', 'dN', '+-', 'SE', 'dS', '+-.1', 'SE.1']


,seq.,seq..1,S,N,t,kappa,omega,dN,+-,SE,dS,+-.1,SE.1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bra033296,AT1G01010,270.8,944.2,0.9290,1.6857,0.5668,0.2646,+-,0.0194,0.4668,+-,0.0602
2,Bra019952,AT1G10270,627.1,1784.9,0.6773,2.5583,0.2564,0.1287,+-,0.0091,0.5020,+-,0.0436
3,Bra018451,AT1G10270,605.9,1797.1,0.7199,2.3475,0.2381,0.1328,+-,0.0092,0.5578,+-,0.0490
4,Bra031706,AT1G10270,622.0,1916.0,0.7699,1.7742,0.1688,0.1163,+-,0.0083,0.6889,+-,0.0628


### Ka/Ks/omega Pre-check

In [42]:
print("=== Ka/Ks/omega Pre-check ===\n")
warnings = 0

# Rename the Bra gene ID column for clarity
# The file has two seq. columns; pandas auto-renames the second to seq..1
evol_df = evol_df.rename(columns={'seq.': 'gene_id', 'seq..1': 'ara_id'})

# --- Duplicate check ---
# 15 duplicate gene IDs are known to exist in this file.
# For each duplicate, keep the row with non-NaN values if one exists.
# Raise a hard error if both rows in a duplicate pair have real (non-NaN)
# conflicting values — that would require manual investigation.
dup_mask = evol_df['gene_id'].duplicated(keep=False)
n_dups = dup_mask.sum()

if n_dups > 0:
    dup_genes = evol_df.loc[dup_mask, 'gene_id'].unique()
    print(f"WARNING: {len(dup_genes)} gene IDs appear more than once "
          f"({n_dups} total rows affected).")

    for gene in dup_genes:
        rows = evol_df[evol_df['gene_id'] == gene]
        has_values = rows[['dN', 'dS', 'omega']].notna().all(axis=1)

        if has_values.sum() > 1:
            # Both rows have real values — flag for manual inspection
            raise ValueError(
                f"CRITICAL: Gene {gene} has {has_values.sum()} rows with "
                f"non-NaN Ka/Ks/omega values. Cannot resolve automatically.\n"
                f"{rows[['gene_id', 'dN', 'dS', 'omega']].to_string()}"
            )
        elif has_values.sum() == 1:
            # One valid row, one NaN row — keep the valid one (handled below)
            pass
        else:
            # Both rows are NaN — both will be dropped (handled below)
            pass

    print(f"  Duplicate resolution: keeping the non-NaN row per gene "
          f"where one exists.")
    # Drop rows where dN/dS/omega are all NaN (these are the bad duplicate rows)
    # Then drop any remaining duplicates keeping the first occurrence
    evol_df = evol_df.dropna(subset=['dN', 'dS', 'omega'])
    evol_df = evol_df.drop_duplicates(subset='gene_id', keep='first')
    print(f"  Rows after duplicate resolution: {len(evol_df)}")
else:
    print("✓ No duplicate gene_ids detected")

# Final duplicate hard check
remaining_dups = evol_df['gene_id'].duplicated().sum()
if remaining_dups > 0:
    raise ValueError(
        f"CRITICAL: {remaining_dups} duplicate gene_ids remain after "
        f"deduplication. Manual investigation required."
    )
print(f"✓ No duplicate gene_ids after resolution\n")

# --- Value range checks ---
# Ka (dN): must be >= 0. Negative values are computational artifacts.
n_neg_ka = (evol_df['dN'] < 0).sum()
if n_neg_ka > 0:
    print(f"WARNING: {n_neg_ka} genes have negative dN (Ka) values. "
          f"Range: {evol_df[evol_df['dN'] < 0]['dN'].min():.6f} to "
          f"{evol_df[evol_df['dN'] < 0]['dN'].max():.6f}.")
    print(f"  These are PAML computational artifacts near zero and will "
          f"be retained as-is.")
    warnings += 1
else:
    print("✓ All dN (Ka) values are >= 0")

# Ks (dS): must be > 0. Zero or near-zero Ks causes omega = inf.
n_zero_ks = (evol_df['dS'] <= 0).sum()
if n_zero_ks > 0:
    raise ValueError(
        f"CRITICAL: {n_zero_ks} genes have dS (Ks) <= 0. "
        f"This would produce infinite omega values."
    )
print("✓ All dS (Ks) values are > 0")

# Omega: must be finite and >= 0
import numpy as np
n_inf_omega  = np.isinf(evol_df['omega']).sum()
n_neg_omega  = (evol_df['omega'] < 0).sum()
if n_inf_omega > 0:
    raise ValueError(
        f"CRITICAL: {n_inf_omega} genes have infinite omega values."
    )
if n_neg_omega > 0:
    raise ValueError(
        f"CRITICAL: {n_neg_omega} genes have negative omega values."
    )
print("✓ All omega values are finite and >= 0\n")

print(f"  dN (Ka) range : {evol_df['dN'].min():.4f} to {evol_df['dN'].max():.4f}")
print(f"  dS (Ks) range : {evol_df['dS'].min():.4f} to {evol_df['dS'].max():.4f}")
print(f"  omega range   : {evol_df['omega'].min():.4f} to {evol_df['omega'].max():.4f}")

# --- Coverage check ---
file_genes  = set(evol_df['gene_id'])
absent_genes = master_genes - file_genes
if absent_genes:
    print(f"\nWARNING: {len(absent_genes)} master doublet genes absent "
          f"from Ka/Ks file.")
    print(f"  These genes receive NaN for dN, dS, and omega.")
    print(f"  Their pairs will be dropped at the final dropout step.")
    print(f"  Absent IDs: {sorted(list(absent_genes))}")
    warnings += 1
else:
    print("\n✓ All master doublet genes present in Ka/Ks file")

print(f"\n=== Pre-check complete. Total warnings: {warnings} ===")

=== Ka/Ks/omega Pre-check ===



ValueError: CRITICAL: Gene Bra024432 has 2 rows with non-NaN Ka/Ks/omega values. Cannot resolve automatically.
         gene_id      dN      dS   omega
23080  Bra024432  0.1391  0.2958  0.4703
23081  Bra024432  0.1391  0.2958  0.4703

### Ka/Ks/omega Processing and Merge

In [43]:
print("=== Ka/Ks/omega Processing and Merge ===\n")

# Retain only the three model features plus the gene ID
evol_clean = evol_df[['gene_id', 'dN', 'dS', 'omega']].copy()

# Build one renamed copy per subgenome.
# Note: because Ka/Ks/omega are per-gene (each B. rapa gene vs Arabidopsis),
# the same LF gene's values appear in both lf_mf1_pairs and lf_mf2_pairs.
# This is by design — see the data source note in the load cell above.
evol_LF = evol_clean.rename(columns={
    'gene_id': 'LF',
    'dN':      'Ka_LF',
    'dS':      'Ks_LF',
    'omega':   'omega_LF'
})
evol_MF1 = evol_clean.rename(columns={
    'gene_id': 'MF1',
    'dN':      'Ka_MF1',
    'dS':      'Ks_MF1',
    'omega':   'omega_MF1'
})
evol_MF2 = evol_clean.rename(columns={
    'gene_id': 'MF2',
    'dN':      'Ka_MF2',
    'dS':      'Ks_MF2',
    'omega':   'omega_MF2'
})

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, evol_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, evol_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, evol_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, evol_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, evol_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, evol_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after Ka/Ks merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after Ka/Ks merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after Ka/Ks merge, expected 2504."

print("✓ Row counts verified after Ka/Ks/omega merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        for feat in ['Ka', 'Ks', 'omega']:
            col   = f'{feat}_{sg}'
            n_nan = df_pair[col].isna().sum()
            if n_nan > 0:
                print(f"  {label} | {col}: {n_nan} NaNs "
                      f"— genes absent from Ka/Ks file, "
                      f"pairs will be dropped at dropout step")
            else:
                print(f"  ✓  {label} | {col}: no NaNs")

=== Ka/Ks/omega Processing and Merge ===

✓ Row counts verified after Ka/Ks/omega merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | Ka_LF: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf1 | Ks_LF: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf1 | omega_LF: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf1 | Ka_MF1: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf1 | Ks_MF1: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf1 | omega_MF1: 3 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf2 | Ka_LF: 1 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf2 | Ks_LF: 1 NaNs — genes absent from Ka/Ks file, pairs will be dropped at dropout step
  lf_mf2 | omega_LF: 1 NaNs — genes absent from Ka/Ks

In [44]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after Ka/Ks/omega merge")

=== Pair-wise Dropout Preview (after Ka/Ks/omega merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 1932 (45.0%)
    Pairs to keep : 2362 (55.0%)
    Columns driving dropout:
      CHG_up_max_MF1: 463 affected pairs
      CHG_up_ave_MF1: 463 affected pairs
      CG_up_max_MF1: 456 affected pairs
      CG_up_ave_MF1: 456 affected pairs
      CHG_up_ave_LF: 426 affected pairs
      CHG_up_max_LF: 426 affected pairs
      CG_up_max_LF: 421 affected pairs
      CG_up_ave_LF: 421 affected pairs
      CG_down_max_MF1: 405 affected pairs
      CG_down_ave_MF1: 405 affected pairs
      CHG_down_max_MF1: 355 affected pairs
      CHG_down_ave_MF1: 355 affected pairs
      CHH_up_max_MF1: 351 affected pairs
      CHH_up_ave_MF1: 351 affected pairs
      CG_body_ave_MF1: 3

## Location Data

In [45]:
# Load chromosomal location (arm vs pericentromeric) annotations.
# Source: B. rapa v1.5 gene coordinate file with location classification.
# File: Brapa_gene_v1.5_genes-1_sorted_location (tab-separated, no header)
#
# Column layout:
#   col 0: chromosome (A01-A10)
#   col 1: gene start
#   col 2: gene end
#   col 3: gene ID
#   col 4: strand
#   col 5: location ('arm' or 'peri')
#
# Location categories:
#   'arm'  : gene is in a chromosome arm region (higher recombination,
#             lower TE density) — encoded as 1.0 in the model
#   'peri' : gene is in a pericentromeric region (lower recombination,
#             higher TE density, more repeat-rich) — encoded as 0.0
#
# This binary feature captures a structural genomic context that is
# known to correlate with gene expression, recombination rate, and TE
# density — all of which relate to subgenome dominance mechanisms
# described in Cheng et al. (2016).
#
# Coverage: only genes on main chromosomes A01-A10 are present (39,608
# genes). The 170 scaffold-resident master doublet genes are absent and
# receive NaN for this feature. These genes are already excluded from
# the model by missing recombination rate data.

loc_df = pd.read_csv(
    '/orange/meixiazhao/meixiazhao/Brassica/Brapa_V1.5/'
    'Brapa_gene_v1.5_genes-1_sorted_location',
    sep='\t', header=None
)
loc_df = loc_df[[3, 5]].rename(columns={3: 'gene_id', 5: 'location'})

print(f"Location dataframe shape: {loc_df.shape}")
print(f"Unique location values: {sorted(loc_df['location'].unique())}")
display(loc_df.head())

Location dataframe shape: (39608, 2)
Unique location values: ['arm', 'peri']


,gene_id,location
0,Bra011902,arm
1,Bra011901,arm
2,Bra011900,arm
3,Bra011899,arm
4,Bra011898,arm


### Location Data Pre-check

In [46]:
print("=== Chromosomal Location Pre-check ===\n")
warnings = 0

# --- Duplicate check ---
dups = loc_df['gene_id'].duplicated()
if dups.any():
    raise ValueError(
        f"CRITICAL: {dups.sum()} duplicate gene_ids in location file.\n"
        f"Affected IDs: {loc_df.loc[dups, 'gene_id'].unique().tolist()}"
    )
print("✓ No duplicate gene_ids detected\n")

# --- Valid values check ---
# Only 'arm' and 'peri' are permitted
valid_locs = {'arm', 'peri'}
unexpected = set(loc_df['location'].unique()) - valid_locs
if unexpected:
    raise ValueError(
        f"CRITICAL: Unexpected location values detected: {unexpected}.\n"
        f"Only 'arm' and 'peri' are valid."
    )
print(f"✓ All location values are valid ('arm' or 'peri')")
print(f"  arm : {(loc_df['location'] == 'arm').sum()} genes")
print(f"  peri: {(loc_df['location'] == 'peri').sum()} genes\n")

# --- Coverage check ---
file_genes   = set(loc_df['gene_id'])
absent_genes = master_genes - file_genes
if absent_genes:
    print(f"INFO: {len(absent_genes)} master doublet genes absent from "
          f"location file.")
    print(f"  All absent genes are on scaffold chromosomes — location")
    print(f"  classification is only defined for main chromosomes A01-A10.")
    print(f"  These genes are already excluded from the model by missing")
    print(f"  recombination rate data. NaN here causes no additional dropout.")
    print(f"  Absent IDs (first 10): "
          f"{sorted(list(absent_genes))[:10]}"
          f"{'...' if len(absent_genes) > 10 else ''}\n")
    warnings += 1
else:
    print("✓ All master doublet genes present in location file\n")

print(f"=== Pre-check complete. Total warnings: {warnings} ===")

=== Chromosomal Location Pre-check ===

✓ No duplicate gene_ids detected

✓ All location values are valid ('arm' or 'peri')
  arm : 33325 genes
  peri: 6283 genes

INFO: 170 master doublet genes absent from location file.
  All absent genes are on scaffold chromosomes — location
  classification is only defined for main chromosomes A01-A10.
  These genes are already excluded from the model by missing
  recombination rate data. NaN here causes no additional dropout.
  Absent IDs (first 10): ['Bra034482', 'Bra034494', 'Bra034495', 'Bra034504', 'Bra034506', 'Bra034515', 'Bra034519', 'Bra034522', 'Bra035332', 'Bra035335']...

=== Pre-check complete. Total warnings: 1 ===


### Location Data Processing and Merge

In [47]:
print("=== Chromosomal Location Processing and Merge ===\n")

# Encode location as binary numeric before merging.
# 'arm'  -> 1.0 (chromosome arm, higher recombination)
# 'peri' -> 0.0 (pericentromeric, lower recombination)
# Encoding is applied here rather than after the pairwise concat step
# (as in the original draft) to keep all feature encoding in the
# feature-specific section and make the final dataset values explicit.
loc_df['location'] = loc_df['location'].map({'arm': 1.0, 'peri': 0.0})

assert loc_df['location'].isin([0.0, 1.0, float('nan')]).all(), \
    "CRITICAL: Unexpected values in location column after encoding."

print(f"✓ Location encoded: arm=1.0 ({(loc_df['location'] == 1.0).sum()} genes), "
      f"peri=0.0 ({(loc_df['location'] == 0.0).sum()} genes)")

# Build per-subgenome merge frames
loc_LF = loc_df.rename(columns={
    'gene_id':  'LF',
    'location': 'location_LF'
})
loc_MF1 = loc_df.rename(columns={
    'gene_id':  'MF1',
    'location': 'location_MF1'
})
loc_MF2 = loc_df.rename(columns={
    'gene_id':  'MF2',
    'location': 'location_MF2'
})

# --- Merge into lf_mf1_pairs ---
lf_mf1_pairs = pd.merge(lf_mf1_pairs, loc_LF,  on='LF',  how='left')
lf_mf1_pairs = pd.merge(lf_mf1_pairs, loc_MF1, on='MF1', how='left')

# --- Merge into lf_mf2_pairs ---
lf_mf2_pairs = pd.merge(lf_mf2_pairs, loc_LF,  on='LF',  how='left')
lf_mf2_pairs = pd.merge(lf_mf2_pairs, loc_MF2, on='MF2', how='left')

# --- Merge into mf1_mf2_pairs ---
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, loc_MF1, on='MF1', how='left')
mf1_mf2_pairs = pd.merge(mf1_mf2_pairs, loc_MF2, on='MF2', how='left')

# --- Hard row count checkpoints ---
assert len(lf_mf1_pairs)  == 4294, \
    f"CRITICAL: lf_mf1_pairs has {len(lf_mf1_pairs)} rows after location merge, expected 4294."
assert len(lf_mf2_pairs)  == 3666, \
    f"CRITICAL: lf_mf2_pairs has {len(lf_mf2_pairs)} rows after location merge, expected 3666."
assert len(mf1_mf2_pairs) == 2504, \
    f"CRITICAL: mf1_mf2_pairs has {len(mf1_mf2_pairs)} rows after location merge, expected 2504."

print("\n✓ Row counts verified after location merge:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

# --- NaN report ---
print("\n--- NaN Report ---")
for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    for sg in [sg1, sg2]:
        col   = f'location_{sg}'
        n_nan = df_pair[col].isna().sum()
        if n_nan > 0:
            print(f"  {label} | {col}: {n_nan} NaNs "
                  f"— scaffold genes, already excluded by missing "
                  f"recombination data, no additional dropout")
        else:
            print(f"  ✓  {label} | {col}: no NaNs")

=== Chromosomal Location Processing and Merge ===

✓ Location encoded: arm=1.0 (33325 genes), peri=0.0 (6283 genes)

✓ Row counts verified after location merge:
  lf_mf1_pairs  : 4294
  lf_mf2_pairs  : 3666
  mf1_mf2_pairs : 2504

--- NaN Report ---
  lf_mf1 | location_LF: 31 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout
  lf_mf1 | location_MF1: 51 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout
  lf_mf2 | location_LF: 39 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout
  lf_mf2 | location_MF2: 22 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout
  mf1_mf2 | location_MF1: 52 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout
  mf1_mf2 | location_MF2: 19 NaNs — scaffold genes, already excluded by missing recombination data, no additional dropout


In [48]:
dropout_preview(lf_mf1_pairs, lf_mf2_pairs, mf1_mf2_pairs,
                label="after location merge")

=== Pair-wise Dropout Preview (after location merge) ===
NaN-exempt columns: ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Total pairs   : 4294
    Pairs to drop : 1932 (45.0%)
    Pairs to keep : 2362 (55.0%)
    Columns driving dropout:
      CHG_up_ave_MF1: 463 affected pairs
      CHG_up_max_MF1: 463 affected pairs
      CG_up_ave_MF1: 456 affected pairs
      CG_up_max_MF1: 456 affected pairs
      CHG_up_max_LF: 426 affected pairs
      CHG_up_ave_LF: 426 affected pairs
      CG_up_ave_LF: 421 affected pairs
      CG_up_max_LF: 421 affected pairs
      CG_down_max_MF1: 405 affected pairs
      CG_down_ave_MF1: 405 affected pairs
      CHG_down_ave_MF1: 355 affected pairs
      CHG_down_max_MF1: 355 affected pairs
      CHH_up_ave_MF1: 351 affected pairs
      CHH_up_max_MF1: 351 affected pairs
      CG_body_ave_MF1: 344 

## Final Output Processing

### Output Directory Setup

In [49]:
import os

OUTPUT_DIR = (
    '/blue/meixiazhao/laylaaschuster/brapaGDM/'
    'final_brapaMLpreprocess/final_featuresets'
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets


### Final Pairwise Dropout

In [50]:
print("=== Final Pair-wise Dropout ===\n")
print("Dropping pairs with NaN in any required feature column.")
print(f"NaN-exempt columns (retained): {sorted(NAN_EXEMPT_COLS)}\n")

metadata_cols = {
    'gene_num', 'ara_paralog', 'AKBr', 'ACK_block', 'LF', 'MF1', 'MF2'
}

dropped_counts = {}

for df_pair, label in [
    (lf_mf1_pairs,  'lf_mf1_pairs'),
    (lf_mf2_pairs,  'lf_mf2_pairs'),
    (mf1_mf2_pairs, 'mf1_mf2_pairs'),
]:
    n_before = len(df_pair)

    dropout_cols = [
        col for col in df_pair.columns
        if col not in metadata_cols and col not in NAN_EXEMPT_COLS
    ]
    drop_mask = df_pair[dropout_cols].isna().any(axis=1)
    n_drop    = drop_mask.sum()
    dropped_counts[label] = n_drop

    print(f"  {label}:")
    print(f"    Pairs before dropout : {n_before}")
    print(f"    Pairs dropped        : {n_drop} ({100 * n_drop / n_before:.1f}%)")
    print(f"    Pairs retained       : {n_before - n_drop}")
    print()

lf_mf1_pairs  = lf_mf1_pairs[
    ~lf_mf1_pairs[
        [c for c in lf_mf1_pairs.columns
         if c not in metadata_cols and c not in NAN_EXEMPT_COLS]
    ].isna().any(axis=1)
].reset_index(drop=True)

lf_mf2_pairs  = lf_mf2_pairs[
    ~lf_mf2_pairs[
        [c for c in lf_mf2_pairs.columns
         if c not in metadata_cols and c not in NAN_EXEMPT_COLS]
    ].isna().any(axis=1)
].reset_index(drop=True)

mf1_mf2_pairs = mf1_mf2_pairs[
    ~mf1_mf2_pairs[
        [c for c in mf1_mf2_pairs.columns
         if c not in metadata_cols and c not in NAN_EXEMPT_COLS]
    ].isna().any(axis=1)
].reset_index(drop=True)

# Hard assertions on retained pair counts — these are the definitive
# pair counts for the B. rapa subgenome dominance model
print("Retained pair counts after dropout:")
print(f"  lf_mf1_pairs  : {len(lf_mf1_pairs)}")
print(f"  lf_mf2_pairs  : {len(lf_mf2_pairs)}")
print(f"  mf1_mf2_pairs : {len(mf1_mf2_pairs)}")

=== Final Pair-wise Dropout ===

Dropping pairs with NaN in any required feature column.
NaN-exempt columns (retained): ['downstream_distance_LF', 'downstream_distance_MF1', 'downstream_distance_MF2', 'tau_LF', 'tau_MF1', 'tau_MF2', 'upstream_distance_LF', 'upstream_distance_MF1', 'upstream_distance_MF2']

  lf_mf1_pairs:
    Pairs before dropout : 4294
    Pairs dropped        : 1932 (45.0%)
    Pairs retained       : 2362

  lf_mf2_pairs:
    Pairs before dropout : 3666
    Pairs dropped        : 1673 (45.6%)
    Pairs retained       : 1993

  mf1_mf2_pairs:
    Pairs before dropout : 2504
    Pairs dropped        : 1213 (48.4%)
    Pairs retained       : 1291

Retained pair counts after dropout:
  lf_mf1_pairs  : 2362
  lf_mf2_pairs  : 1993
  mf1_mf2_pairs : 1291


###  Gene ID Overlap Check and Group Assignment

In [51]:
print("=== Gene ID Overlap Check ===\n")
print("Verifying no gene ID appears in both subgenome columns of the same pair.")
print("A gene cannot be its own pair partner — this would corrupt WGD labels.\n")

for df_pair, label, sg1, sg2 in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2'),
]:
    overlap = set(df_pair[sg1]) & set(df_pair[sg2])
    if overlap:
        raise ValueError(
            f"CRITICAL: {len(overlap)} gene IDs appear in both {sg1} and "
            f"{sg2} columns of {label}. WGD label assignment will be "
            f"incorrect.\nAffected IDs: {sorted(list(overlap))}"
        )
    print(f"  ✓ {label}: no gene ID overlap between {sg1} and {sg2} columns")

print("\n✓ WGD label assignment is safe to proceed for all three comparisons")

# --- Group assignment (on wide format, before stacking) ---
# Groups are defined by the chromosomal location combination of the pair:
#   Group I   : both genes in chromosome arm regions (arm-arm)
#   Group II  : both genes in pericentromeric regions (peri-peri)
#   Group III : dominant subgenome gene in arm, non-dominant in peri
#   Group IV  : dominant subgenome gene in peri, non-dominant in arm
#
# Groups are assigned here at the pair level so that after stacking,
# both members of each pair carry the same group label.
# Pairs where either gene has NaN location were already dropped at dropout.

def assign_groups(df, sg1, sg2):
    """
    Assign location-based groups to gene pairs.

    Parameters
    ----------
    df  : pd.DataFrame — pairwise dataframe with location columns
    sg1 : str — dominant subgenome label (e.g. 'LF' or 'MF1')
    sg2 : str — non-dominant subgenome label (e.g. 'MF1' or 'MF2')

    Returns
    -------
    pd.Series of group labels ('I', 'II', 'III', 'IV')
    """
    loc1 = df[f'location_{sg1}']
    loc2 = df[f'location_{sg2}']

    conditions = [
        (loc1 == 1.0) & (loc2 == 1.0),   # arm–arm
        (loc1 == 0.0) & (loc2 == 0.0),   # peri–peri
        (loc1 == 1.0) & (loc2 == 0.0),   # arm–peri
        (loc1 == 0.0) & (loc2 == 1.0),   # peri–arm
    ]
    choices = ['I', 'II', 'III', 'IV']

    import numpy as np
    group = np.select(conditions, choices, default=None)
    n_none = (group == None).sum()
    if n_none > 0:
        raise ValueError(
            f"CRITICAL: {n_none} pairs could not be assigned a group. "
            f"This indicates unexpected NaN values in location columns "
            f"that should have been caught at dropout."
        )
    return pd.Series(group, index=df.index)


lf_mf1_pairs['group']  = assign_groups(lf_mf1_pairs,  'LF',  'MF1')
lf_mf2_pairs['group']  = assign_groups(lf_mf2_pairs,  'LF',  'MF2')
mf1_mf2_pairs['group'] = assign_groups(mf1_mf2_pairs, 'MF1', 'MF2')

print("\n=== Group Assignment Counts ===")
for df_pair, label in [
    (lf_mf1_pairs,  'lf_mf1'),
    (lf_mf2_pairs,  'lf_mf2'),
    (mf1_mf2_pairs, 'mf1_mf2'),
]:
    print(f"\n  {label}:")
    for grp in ['I', 'II', 'III', 'IV']:
        n = (df_pair['group'] == grp).sum()
        print(f"    Group {grp}: {n} pairs")

=== Gene ID Overlap Check ===

Verifying no gene ID appears in both subgenome columns of the same pair.
A gene cannot be its own pair partner — this would corrupt WGD labels.

  ✓ lf_mf1: no gene ID overlap between LF and MF1 columns
  ✓ lf_mf2: no gene ID overlap between LF and MF2 columns
  ✓ mf1_mf2: no gene ID overlap between MF1 and MF2 columns

✓ WGD label assignment is safe to proceed for all three comparisons

=== Group Assignment Counts ===

  lf_mf1:
    Group I: 2036 pairs
    Group II: 30 pairs
    Group III: 206 pairs
    Group IV: 90 pairs

  lf_mf2:
    Group I: 1588 pairs
    Group II: 32 pairs
    Group III: 278 pairs
    Group IV: 95 pairs

  mf1_mf2:
    Group I: 1017 pairs
    Group II: 57 pairs
    Group III: 128 pairs
    Group IV: 89 pairs


### Wide-to-Long Stacking Function

In [52]:
def stack_pairwise(df, sg1, sg2, wgd_label_sg1=0, wgd_label_sg2=1):
    """
    Convert a pairwise (wide-format) dataframe to a gene-level
    (long-format) dataframe by stacking both subgenome members
    of each pair into individual rows.

    The resulting dataframe has 2 × n_pairs rows. Each row represents
    one gene with its feature values, WGD label, group assignment, and
    chromosomal location. This format is required for model training.

    WGD label convention (mirrors maize model):
        sg1 (dominant subgenome) = 0
        sg2 (non-dominant subgenome) = 1

    Parameters
    ----------
    df           : pd.DataFrame — wide pairwise dataframe after dropout
    sg1          : str — dominant subgenome column name (e.g. 'LF')
    sg2          : str — non-dominant subgenome column name (e.g. 'MF1')
    wgd_label_sg1 : int — WGD label for sg1 genes (default 0)
    wgd_label_sg2 : int — WGD label for sg2 genes (default 1)

    Returns
    -------
    pd.DataFrame with columns:
        gene_id, WGD, group, location, <feature columns...>
    """
    # Identify feature columns for each subgenome
    # All columns ending with _{sg1} or _{sg2} are feature columns
    sg1_feat_cols = [c for c in df.columns if c.endswith(f'_{sg1}')]
    sg2_feat_cols = [c for c in df.columns if c.endswith(f'_{sg2}')]

    # Base feature names (strip subgenome suffix)
    suffix_len_sg1 = len(f'_{sg1}')
    suffix_len_sg2 = len(f'_{sg2}')
    sg1_base = [c[:-suffix_len_sg1] for c in sg1_feat_cols]
    sg2_base = [c[:-suffix_len_sg2] for c in sg2_feat_cols]

    # Confirm both subgenomes have the same set of base features
    if set(sg1_base) != set(sg2_base):
        only_sg1 = set(sg1_base) - set(sg2_base)
        only_sg2 = set(sg2_base) - set(sg1_base)
        raise ValueError(
            f"CRITICAL: Feature column mismatch between {sg1} and {sg2}.\n"
            f"Features only in {sg1}: {only_sg1}\n"
            f"Features only in {sg2}: {only_sg2}"
        )

    # Build sg1 (dominant) half — one row per pair
    sg1_df = df[[sg1, 'group'] + sg1_feat_cols].copy()
    sg1_df = sg1_df.rename(
        columns={sg1: 'gene_id',
                 **{c: c[:-suffix_len_sg1] for c in sg1_feat_cols}}
    )
    sg1_df['WGD'] = wgd_label_sg1

    # Build sg2 (non-dominant) half — one row per pair
    sg2_df = df[[sg2, 'group'] + sg2_feat_cols].copy()
    sg2_df = sg2_df.rename(
        columns={sg2: 'gene_id',
                 **{c: c[:-suffix_len_sg2] for c in sg2_feat_cols}}
    )
    sg2_df['WGD'] = wgd_label_sg2

    # Stack vertically — preserve pair ordering (sg1 then sg2)
    stacked = pd.concat([sg1_df, sg2_df], ignore_index=True)

    # Move gene_id, WGD, group to front for readability
    front_cols = ['gene_id', 'WGD', 'group']
    remaining  = [c for c in stacked.columns if c not in front_cols]
    stacked    = stacked[front_cols + remaining]

    return stacked

### Process, Export, and Summarise All Three Pairwise Datasets

In [53]:
print("=== Processing, Exporting, and Summarising All Pairwise Datasets ===\n")

# Columns that identify genes but are not model features
# These are retained in the all_columns version but dropped for model input
id_group_cols = ['gene_id', 'group']

# Summary table for all output shapes
shape_summary = []

for df_pair, label, sg1, sg2, pair_name in [
    (lf_mf1_pairs,  'lf_mf1',  'LF',  'MF1', 'LF–MF1'),
    (lf_mf2_pairs,  'lf_mf2',  'LF',  'MF2', 'LF–MF2'),
    (mf1_mf2_pairs, 'mf1_mf2', 'MF1', 'MF2', 'MF1–MF2'),
]:
    print(f"--- {pair_name} ---")

    # Stack wide-to-long
    stacked = stack_pairwise(df_pair, sg1, sg2,
                             wgd_label_sg1=0, wgd_label_sg2=1)

    # Verify WGD balance — each subgenome should have the same number of genes
    wgd_counts = stacked['WGD'].value_counts()
    n_sg1 = wgd_counts.get(0, 0)
    n_sg2 = wgd_counts.get(1, 0)
    assert n_sg1 == n_sg2, \
        f"CRITICAL: {pair_name} WGD imbalance — WGD=0: {n_sg1}, WGD=1: {n_sg2}"
    assert len(stacked) == 2 * len(df_pair), \
        f"CRITICAL: {pair_name} stacked row count {len(stacked)} != " \
        f"2 × {len(df_pair)} pairs"
    print(f"  ✓ Stacked shape: {stacked.shape} "
          f"({len(df_pair)} pairs × 2 = {len(stacked)} genes)")
    print(f"  ✓ WGD balanced: {n_sg1} per subgenome")

    # Save full pairwise dataset
    path_all = os.path.join(OUTPUT_DIR, f'Brapa_{label}_all_columns_final.csv')
    path_feat = os.path.join(OUTPUT_DIR, f'Brapa_{label}_final.csv')

    stacked.to_csv(path_all, index=False)
    stacked.drop(columns=id_group_cols).to_csv(path_feat, index=False)
    print(f"  Saved: Brapa_{label}_all_columns_final.csv")
    print(f"  Saved: Brapa_{label}_final.csv")

    shape_summary.append({
        'comparison': pair_name, 'group': 'ALL',
        'n_pairs': len(df_pair), 'n_genes': len(stacked),
        'n_features': len(stacked.columns) - len(id_group_cols) - 1  # -1 for WGD
    })

    # Verify expected group labels
    observed_groups = set(stacked['group'].unique())
    expected_groups = {'I', 'II', 'III', 'IV'}
    unexpected = observed_groups - expected_groups
    if unexpected:
        raise ValueError(
            f"CRITICAL: Unexpected group labels in {pair_name}: {unexpected}"
        )
    missing = expected_groups - observed_groups
    if missing:
        print(f"  NOTE: Group(s) {missing} have zero pairs in {pair_name}.")

    # Group subsetting and export
    print(f"\n  Group breakdown:")
    for grp in ['I', 'II', 'III', 'IV']:
        df_grp = stacked[stacked['group'] == grp].copy()
        n_pairs_grp = len(df_grp) // 2

        wgd_grp = df_grp['WGD'].value_counts()
        balanced = wgd_grp.get(0, 0) == wgd_grp.get(1, 0)

        path_grp_all  = os.path.join(
            OUTPUT_DIR, f'Brapa_{label}_group{grp}_all_columns_final.csv'
        )
        path_grp_feat = os.path.join(
            OUTPUT_DIR, f'Brapa_{label}_group{grp}_final.csv'
        )
        df_grp.to_csv(path_grp_all, index=False)
        df_grp.drop(columns=id_group_cols).to_csv(path_grp_feat, index=False)

        balance_flag = '✓' if balanced else '⚠ UNBALANCED'
        print(f"    Group {grp}: {n_pairs_grp:>5} pairs "
              f"({len(df_grp):>5} genes) {balance_flag}")
        print(f"      Saved: Brapa_{label}_group{grp}_all_columns_final.csv")
        print(f"      Saved: Brapa_{label}_group{grp}_final.csv")

        shape_summary.append({
            'comparison': pair_name, 'group': grp,
            'n_pairs': n_pairs_grp, 'n_genes': len(df_grp),
            'n_features': len(df_grp.columns) - len(id_group_cols) - 1
        })

    print()

print("=== All files saved ===")

=== Processing, Exporting, and Summarising All Pairwise Datasets ===

--- LF–MF1 ---
  ✓ Stacked shape: (4724, 48) (2362 pairs × 2 = 4724 genes)
  ✓ WGD balanced: 2362 per subgenome
  Saved: Brapa_lf_mf1_all_columns_final.csv
  Saved: Brapa_lf_mf1_final.csv

  Group breakdown:
    Group I:  2036 pairs ( 4072 genes) ✓
      Saved: Brapa_lf_mf1_groupI_all_columns_final.csv
      Saved: Brapa_lf_mf1_groupI_final.csv
    Group II:    30 pairs (   60 genes) ✓
      Saved: Brapa_lf_mf1_groupII_all_columns_final.csv
      Saved: Brapa_lf_mf1_groupII_final.csv
    Group III:   206 pairs (  412 genes) ✓
      Saved: Brapa_lf_mf1_groupIII_all_columns_final.csv
      Saved: Brapa_lf_mf1_groupIII_final.csv
    Group IV:    90 pairs (  180 genes) ✓
      Saved: Brapa_lf_mf1_groupIV_all_columns_final.csv
      Saved: Brapa_lf_mf1_groupIV_final.csv

--- LF–MF2 ---
  ✓ Stacked shape: (3986, 48) (1993 pairs × 2 = 3986 genes)
  ✓ WGD balanced: 1993 per subgenome
  Saved: Brapa_lf_mf2_all_columns_final.c

### Output Shape Summary

In [54]:
print("=== Output Shape Summary ===\n")
print("This table shows the number of gene pairs available in each output file.")
print("Only Group I (arm-arm pairs) is expected to be large enough for the")
print("ML pipeline. Groups II-IV are saved for reference.\n")

summary_df = pd.DataFrame(shape_summary)
summary_df = summary_df.set_index(['comparison', 'group'])

print(f"{'Comparison':<12} {'Group':<8} {'n_pairs':>10} "
      f"{'n_genes':>10} {'n_features':>12}")
print("-" * 56)
for (comparison, group), row in summary_df.iterrows():
    flag = ' ← model-ready' if group == 'I' else ''
    print(f"{comparison:<12} {'Group ' + group:<8} {row['n_pairs']:>10} "
          f"{row['n_genes']:>10} {row['n_features']:>12}{flag}")

print()
print("Files saved to:")
print(f"  {OUTPUT_DIR}")
print(f"\nFile types per dataset:")
print(f"  *_all_columns_final.csv : gene_id, WGD, group, all features "
      f"(for traceability)")
print(f"  *_final.csv             : WGD label + feature columns only "
      f"(for model input)")

=== Output Shape Summary ===

This table shows the number of gene pairs available in each output file.
Only Group I (arm-arm pairs) is expected to be large enough for the
ML pipeline. Groups II-IV are saved for reference.

Comparison   Group       n_pairs    n_genes   n_features
--------------------------------------------------------
LF–MF1       Group ALL       2362       4724           45
LF–MF1       Group I        2036       4072           45 ← model-ready
LF–MF1       Group II         30         60           45
LF–MF1       Group III        206        412           45
LF–MF1       Group IV         90        180           45
LF–MF2       Group ALL       1993       3986           45
LF–MF2       Group I        1588       3176           45 ← model-ready
LF–MF2       Group II         32         64           45
LF–MF2       Group III        278        556           45
LF–MF2       Group IV         95        190           45
MF1–MF2      Group ALL       1291       2582           45
MF1